# 07 - Model Training

Objective

This notebook prepares the modeling datasets, trains baseline and tuned candidate models, and selects the candidate final model for downstream evaluation.

The workflow preserves temporal ordering, prevents target leakage, compares Logistic Regression, Random Forest, and XGBoost, and persists modeling checkpoints for the evaluation and explainability notebooks.

Holdout testing, calibration analysis, fitted-model artifact saving, and model explainability are performed in Notebooks 08 and 09.

#### Load project configuration


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration

from config import project_config as cfg

print("Project configuration loaded successfully.")


#### Load and validate the feature dataset

The model-training process begins by loading the managed `flights_features` Delta table produced by the Feature Engineering notebook.

Before splitting or modelling, the dataset is validated to confirm that:

- The required Unity Catalog table exists
- The target variable is available
- The flight date is stored as a valid date
- All required schedule-time predictors are present
- The dataset contains records suitable for chronological splitting


In [0]:
from __future__ import annotations

import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T



FEATURE_TABLE = cfg.FEATURES_TABLE
TARGET_COLUMN = cfg.TARGET_COLUMN
DATE_COLUMN = cfg.FLIGHT_DATE_COLUMN


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the feature-engineering notebook (06) before continuing."
        )


require_table(FEATURE_TABLE)

df_features: DataFrame = spark.table(FEATURE_TABLE)

required_columns = cfg.MODEL_TRAINING_REQUIRED_COLUMNS

missing_columns = sorted(required_columns - set(df_features.columns))

if missing_columns:
    raise ValueError(
        "Model-training validation failed. "
        f"Missing required columns: {missing_columns}"
    )

feature_row_count = df_features.count()
feature_column_count = len(df_features.columns)

date_type = df_features.schema[DATE_COLUMN].dataType

if not isinstance(date_type, T.DateType):
    raise TypeError(
        f"{DATE_COLUMN} must be a Spark date column, "
        f"but found {date_type.simpleString()}."
    )

print("Feature dataset loaded and validated successfully.")
print(f"Source table: {FEATURE_TABLE}")
print(f"Total records: {feature_row_count:,}")
print(f"Total columns: {feature_column_count}")
print(f"Prediction target: {TARGET_COLUMN}")
print(f"Date column type: {date_type.simpleString()}")

#### Date range and target distribution

Before defining the chronological training, validation, and test periods, the feature dataset is examined to confirm its available date range and the distribution of the binary target variable.

This review supports two important modelling decisions:

- Selecting non-overlapping chronological split periods
- Assessing whether the delayed and on-time classes are imbalanced

No records are modified during this analysis.


In [0]:
dataset_profile = (
    df_features
    .select(
        F.min("FL_DATE").alias("MIN_FL_DATE"),
        F.max("FL_DATE").alias("MAX_FL_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 0, 1).otherwise(0)
        ).alias("ON_TIME_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 1, 1).otherwise(0)
        ).alias("DELAYED_RECORDS"),
    )
    .withColumn(
        "ON_TIME_PERCENTAGE",
        F.round(
            F.col("ON_TIME_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
    .withColumn(
        "DELAYED_PERCENTAGE",
        F.round(
            F.col("DELAYED_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
)

display(dataset_profile)

#### Create chronological train, validation, and test splits

The dataset is divided chronologically rather than randomly because the model is intended to predict future flight-delay risk from historical observations.

The split periods are defined as follows:

- **Training period:** January 1, 2025 to August 31, 2025
- **Validation period:** September 1, 2025 to October 31, 2025
- **Test period:** November 1, 2025 to December 31, 2025

This design ensures that later flight outcomes are not used to train models evaluated on earlier periods. The validation dataset will support model and hyperparameter selection, while the test dataset will remain untouched until final evaluation.


In [0]:
TRAIN_END_DATE = cfg.TRAIN_END_DATE
VALIDATION_START_DATE = cfg.VALIDATION_START_DATE
VALIDATION_END_DATE = cfg.VALIDATION_END_DATE
TEST_START_DATE = cfg.TEST_START_DATE

df_train = df_features.filter(
    F.col("FL_DATE") <= F.to_date(F.lit(TRAIN_END_DATE))
)

df_validation = df_features.filter(
    (F.col("FL_DATE") >= F.to_date(F.lit(VALIDATION_START_DATE)))
    & (F.col("FL_DATE") <= F.to_date(F.lit(VALIDATION_END_DATE)))
)

df_test = df_features.filter(
    F.col("FL_DATE") >= F.to_date(F.lit(TEST_START_DATE))
)

split_summary = (
    df_train.select(
        F.lit("TRAIN").alias("DATASET"),
        F.min("FL_DATE").alias("MIN_DATE"),
        F.max("FL_DATE").alias("MAX_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
    )
    .unionByName(
        df_validation.select(
            F.lit("VALIDATION").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .unionByName(
        df_test.select(
            F.lit("TEST").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .withColumn(
        "DELAY_PERCENTAGE",
        F.round(F.col("DELAY_RATE") * 100, 4),
    )
    .drop("DELAY_RATE")
)

display(split_summary)

#### Split validation summary

The chronological split produced three non-overlapping datasets whose combined record count matches the complete feature dataset.

The target distribution varies across the periods:

- The training period has a delay rate of approximately 22.91%.
- The validation period has a lower delay rate of approximately 18.55%.
- The test period has a higher delay rate of approximately 23.71%.

This variation reflects temporal changes in airline operations and confirms the importance of evaluating the model on future periods rather than using a random split.


#### Engineer leakage-safe historical features

Historical performance features summarize prior delay behaviour for airlines, airports, and routes.

To prevent target leakage:

- Historical features for training records use only flights from earlier dates.
- The current record and later training outcomes are excluded.
- Validation and test mappings will be calculated from the training period only.
- A global training delay rate will be used when insufficient historical observations are available.

The first feature created is `AIRLINE_HIST_DELAY_RATE`, representing an airline's smoothed arrival-delay rate before the current flight date.


In [0]:
from pyspark.sql.window import Window


# Daily airline-level delay statistics within the training period
airline_daily_stats = (
    df_train
    .groupBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias("DAILY_DELAY_COUNT"),
    )
)

# Use only dates before the current flight date
airline_history_window = (
    Window
    .partitionBy("OP_UNIQUE_CARRIER")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

airline_daily_history = (
    airline_daily_stats
    .withColumn(
        "AIRLINE_PRIOR_FLIGHTS",
        F.sum("DAILY_FLIGHT_COUNT").over(airline_history_window),
    )
    .withColumn(
        "AIRLINE_PRIOR_DELAYS",
        F.sum("DAILY_DELAY_COUNT").over(airline_history_window),
    )
)

display(
    airline_daily_history
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "DAILY_FLIGHT_COUNT",
        "DAILY_DELAY_COUNT",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
    )
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .limit(30)
)

#### Historical airline delay rate

The `AIRLINE_HIST_DELAY_RATE` feature represents an airline's arrival-delay rate using only flights from earlier training dates.

A smoothed estimate is used to prevent unstable rates when an airline has limited prior observations. The overall training delay rate serves as the prior and as the fallback value for the first available date, when no earlier airline history exists.

The current date's outcomes are excluded from the calculation.


In [0]:
# Overall delay rate from the training period.
# This is used as the smoothing prior and first-date fallback.
global_training_delay_rate = (
    df_train
    .select(F.avg(F.col("ARR_DEL15").cast("double")).alias("GLOBAL_DELAY_RATE"))
    .first()["GLOBAL_DELAY_RATE"]
)

# Controls how strongly low-volume airline histories are pulled
# toward the global training delay rate.
SMOOTHING_STRENGTH = 100.0

airline_history_features = (
    airline_daily_history
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("AIRLINE_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("AIRLINE_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

# Join the leakage-safe airline history to every training flight.
df_train_hist = (
    df_train
    .join(
        airline_history_features,
        on=["OP_UNIQUE_CARRIER", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("AIRLINE_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Global training delay rate: {global_training_delay_rate:.6f}")
print(f"Smoothing strength: {SMOOTHING_STRENGTH:.0f}")
print(f"Training rows after join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
        "AIRLINE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .limit(30)
)

#### Historical origin-airport delay rate

The `ORIGIN_HIST_DELAY_RATE` feature represents the prior arrival-delay rate of flights departing from each origin airport.

For every training record, only flights from earlier dates at the same origin airport are included. The current date and all future records are excluded to prevent target leakage.

A smoothed estimate is used, with the global training delay rate serving as the prior and as the fallback when no earlier airport history exists.


In [0]:
# Daily origin-airport delay statistics within the training period
origin_daily_stats = (
    df_train
    .groupBy(
        "ORIGIN",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_ORIGIN_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_ORIGIN_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
origin_history_window = (
    Window
    .partitionBy("ORIGIN")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

origin_history_features = (
    origin_daily_stats
    .withColumn(
        "ORIGIN_PRIOR_FLIGHTS",
        F.sum("DAILY_ORIGIN_FLIGHT_COUNT").over(origin_history_window),
    )
    .withColumn(
        "ORIGIN_PRIOR_DELAYS",
        F.sum("DAILY_ORIGIN_DELAY_COUNT").over(origin_history_window),
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("ORIGIN_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("ORIGIN_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "FL_DATE",
        "ORIGIN_PRIOR_FLIGHTS",
        "ORIGIN_PRIOR_DELAYS",
        "ORIGIN_HIST_DELAY_RATE",
    )
)

# Join origin history onto the training dataset that already contains
# AIRLINE_HIST_DELAY_RATE
df_train_hist = (
    df_train_hist
    .join(
        origin_history_features,
        on=["ORIGIN", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ORIGIN_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after origin join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "ORIGIN",
        "FL_DATE",
        "ORIGIN_PRIOR_FLIGHTS",
        "ORIGIN_PRIOR_DELAYS",
        "ORIGIN_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "ORIGIN",
        "FL_DATE",
    )
    .limit(30)
)

#### Historical destination-airport delay rate

The `DEST_HIST_DELAY_RATE` feature represents the prior arrival-delay rate of flights travelling to each destination airport.

For every training record, only flights from earlier dates with the same destination airport are included. The current date and all future observations are excluded to prevent target leakage.

A smoothed estimate is used, with the global training delay rate serving as the prior and as the fallback when no earlier destination history exists.


In [0]:
# Daily destination-airport delay statistics within the training period
dest_daily_stats = (
    df_train
    .groupBy(
        "DEST",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_DEST_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_DEST_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
dest_history_window = (
    Window
    .partitionBy("DEST")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

dest_history_features = (
    dest_daily_stats
    .withColumn(
        "DEST_PRIOR_FLIGHTS",
        F.sum("DAILY_DEST_FLIGHT_COUNT").over(dest_history_window),
    )
    .withColumn(
        "DEST_PRIOR_DELAYS",
        F.sum("DAILY_DEST_DELAY_COUNT").over(dest_history_window),
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("DEST_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("DEST_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "DEST",
        "FL_DATE",
        "DEST_PRIOR_FLIGHTS",
        "DEST_PRIOR_DELAYS",
        "DEST_HIST_DELAY_RATE",
    )
)

# Join destination history onto the training dataset
df_train_hist = (
    df_train_hist
    .join(
        dest_history_features,
        on=["DEST", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        F.coalesce(
            F.col("DEST_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after destination join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "DEST",
        "FL_DATE",
        "DEST_PRIOR_FLIGHTS",
        "DEST_PRIOR_DELAYS",
        "DEST_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "DEST",
        "FL_DATE",
    )
    .limit(30)
)

#### Historical route delay rate

The `ROUTE_HIST_DELAY_RATE` feature represents the prior arrival-delay rate for each origin–destination route.

For every training record, only flights from earlier dates on the same route are included. The current date and all future observations are excluded to prevent target leakage.

Because some routes have limited historical volume, a smoothed estimate is used. The global training delay rate serves as the prior and as the fallback when no earlier route history exists.


In [0]:
# Daily route-level delay statistics within the training period
route_daily_stats = (
    df_train
    .groupBy(
        "ORIGIN",
        "DEST",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_ROUTE_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_ROUTE_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
route_history_window = (
    Window
    .partitionBy("ORIGIN", "DEST")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

route_history_features = (
    route_daily_stats
    .withColumn(
        "ROUTE_PRIOR_FLIGHTS",
        F.sum("DAILY_ROUTE_FLIGHT_COUNT").over(route_history_window),
    )
    .withColumn(
        "ROUTE_PRIOR_DELAYS",
        F.sum("DAILY_ROUTE_DELAY_COUNT").over(route_history_window),
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("ROUTE_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("ROUTE_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "DEST",
        "FL_DATE",
        "ROUTE_PRIOR_FLIGHTS",
        "ROUTE_PRIOR_DELAYS",
        "ROUTE_HIST_DELAY_RATE",
    )
)

# Join route history onto the training dataset
df_train_hist = (
    df_train_hist
    .join(
        route_history_features,
        on=["ORIGIN", "DEST", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ROUTE_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after route join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "ORIGIN",
        "DEST",
        "FL_DATE",
        "ROUTE_PRIOR_FLIGHTS",
        "ROUTE_PRIOR_DELAYS",
        "ROUTE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "ORIGIN",
        "DEST",
        "FL_DATE",
    )
    .limit(30)
)

#### Apply training-only historical features to validation and test data

Historical mappings for airlines, origin airports, destination airports, and routes are calculated exclusively from the training period.

These fixed training-period mappings are then joined to the validation and test datasets. Neither validation nor test outcomes are used when calculating the historical rates.

For categories not observed during training, the global training delay rate is used as a fallback.


In [0]:
# -------------------------------------------------------
# Create historical mappings from training data only
# -------------------------------------------------------

airline_training_map = (
    df_train
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.count("*").alias("AIRLINE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "AIRLINE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.col("AIRLINE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("AIRLINE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

origin_training_map = (
    df_train
    .groupBy("ORIGIN")
    .agg(
        F.count("*").alias("ORIGIN_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ORIGIN_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        (
            F.col("ORIGIN_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ORIGIN_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "ORIGIN_HIST_DELAY_RATE",
    )
)

dest_training_map = (
    df_train
    .groupBy("DEST")
    .agg(
        F.count("*").alias("DEST_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DEST_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        (
            F.col("DEST_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("DEST_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "DEST",
        "DEST_HIST_DELAY_RATE",
    )
)

route_training_map = (
    df_train
    .groupBy("ORIGIN", "DEST")
    .agg(
        F.count("*").alias("ROUTE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ROUTE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        (
            F.col("ROUTE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ROUTE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "DEST",
        "ROUTE_HIST_DELAY_RATE",
    )
)


def attach_training_history(dataset: DataFrame) -> DataFrame:
    """Attach historical rates calculated exclusively from training data."""
    return (
        dataset
        .join(
            airline_training_map,
            on="OP_UNIQUE_CARRIER",
            how="left",
        )
        .join(
            origin_training_map,
            on="ORIGIN",
            how="left",
        )
        .join(
            dest_training_map,
            on="DEST",
            how="left",
        )
        .join(
            route_training_map,
            on=["ORIGIN", "DEST"],
            how="left",
        )
        .fillna(
            {
                "AIRLINE_HIST_DELAY_RATE": global_training_delay_rate,
                "ORIGIN_HIST_DELAY_RATE": global_training_delay_rate,
                "DEST_HIST_DELAY_RATE": global_training_delay_rate,
                "ROUTE_HIST_DELAY_RATE": global_training_delay_rate,
            }
        )
    )


df_validation_hist = attach_training_history(df_validation)
df_test_hist = attach_training_history(df_test)

print(
    f"Validation rows after historical joins: "
    f"{df_validation_hist.count():,}"
)
print(
    f"Test rows after historical joins: "
    f"{df_test_hist.count():,}"
)

display(
    df_validation_hist
    .select(
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        "AIRLINE_HIST_DELAY_RATE",
        "ORIGIN_HIST_DELAY_RATE",
        "DEST_HIST_DELAY_RATE",
        "ROUTE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .limit(20)
)

#### Validate historical performance features

The historical feature datasets are validated before categorical encoding and model training.

This check confirms that:

- Record counts remain unchanged after historical-feature joins
- No historical delay-rate features contain missing values
- All historical rates fall within the valid probability range of 0 to 1
- Training, validation, and test datasets contain the same historical feature columns


In [0]:
HISTORICAL_RATE_COLUMNS = list(cfg.MODEL_HISTORICAL_RATE_COLUMNS)

historical_validation_rows = []

for dataset_name, dataset in [
    ("TRAIN", df_train_hist),
    ("VALIDATION", df_validation_hist),
    ("TEST", df_test_hist),
]:
    summary_row = (
        dataset
        .select(
            F.lit(dataset_name).alias("DATASET"),
            F.count("*").alias("TOTAL_RECORDS"),
            *[
                F.sum(
                    F.when(F.col(column_name).isNull(), 1).otherwise(0)
                ).alias(f"{column_name}_NULLS")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
            *[
                F.sum(
                    F.when(
                        (F.col(column_name) < 0)
                        | (F.col(column_name) > 1),
                        1,
                    ).otherwise(0)
                ).alias(f"{column_name}_OUT_OF_RANGE")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
        )
    )

    historical_validation_rows.append(summary_row)

historical_validation_summary = historical_validation_rows[0]

for summary_row in historical_validation_rows[1:]:
    historical_validation_summary = (
        historical_validation_summary.unionByName(summary_row)
    )

display(historical_validation_summary)


#### Encode categorical variables

Categorical and numerical predictors are converted into a model-ready sparse feature vector using Spark's `FeatureHasher`.

Feature hashing maps categorical values into a fixed-dimensional numerical representation without fitting and storing large category-indexing models. This approach avoids the Spark Connect ML model-cache limitation encountered with `StringIndexer` and `OneHotEncoder`.

The same deterministic transformation is applied to the training, validation, and test datasets. No validation or test outcomes are used during preprocessing.

Before applying the transformation, the preprocessing configuration is validated to ensure that all required model input columns are present across the chronological training, validation, and test datasets. The resulting hashed datasets retain only `FL_DATE`, the target variable, and the generated `features` vector required by the machine learning algorithms. Each transformed dataset is then validated to ensure that feature hashing has been applied successfully before model training begins.


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from utils.model_training import (
    create_feature_hasher,
    hash_modeling_frame,
    prepare_hist_modeling_frame,
    validate_hist_modeling_frame,
    validate_feature_hasher,
)


CATEGORICAL_COLUMNS = list(cfg.MODEL_CATEGORICAL_COLUMNS)
NUMERICAL_COLUMNS = list(cfg.MODEL_NUMERICAL_COLUMNS)
MODEL_INPUT_COLUMNS = list(cfg.MODEL_INPUT_COLUMNS)
HASH_VECTOR_SIZE = cfg.HASH_VECTOR_SIZE

# Drop intermediate prior-count columns before hashing or checkpoint persistence.
df_train_hist = prepare_hist_modeling_frame(df_train_hist)
df_validation_hist = prepare_hist_modeling_frame(df_validation_hist)
df_test_hist = prepare_hist_modeling_frame(df_test_hist)

for dataframe_name, dataframe in {
    "df_train_hist": df_train_hist,
    "df_validation_hist": df_validation_hist,
    "df_test_hist": df_test_hist,
}.items():
    validate_hist_modeling_frame(dataframe, dataframe_name)

feature_hasher = create_feature_hasher()
validate_feature_hasher(feature_hasher)

df_train_hashed = hash_modeling_frame(df_train_hist, feature_hasher)
df_validation_hashed = hash_modeling_frame(df_validation_hist, feature_hasher)
df_test_hashed = hash_modeling_frame(df_test_hist, feature_hasher)

for dataframe_name, dataframe in {
    "df_train_hashed": df_train_hashed,
    "df_validation_hashed": df_validation_hashed,
    "df_test_hashed": df_test_hashed,
}.items():
    validation_row_count = (
        dataframe
        .select("features")
        .limit(1)
        .count()
    )

    if validation_row_count == 0:
        raise ValueError(
            f"{dataframe_name} contains no rows."
        )

    print(
        f"{dataframe_name} created and validated successfully."
    )

print()
print("Feature-hashing preprocessing configured successfully.")
print(f"Categorical features: {len(CATEGORICAL_COLUMNS)}")
print(f"Numerical features: {len(NUMERICAL_COLUMNS)}")
print(f"Total raw predictors: {len(MODEL_INPUT_COLUMNS)}")
print(f"Hashed vector size: {HASH_VECTOR_SIZE:,}")
print(f"Target: {TARGET_COLUMN}")
print(f"Feature hasher inputs: {feature_hasher.getInputCols()}")

df_train_prepared = df_train_hashed
df_validation_prepared = df_validation_hashed
df_test_prepared = df_test_hashed


#### Apply feature hashing to model datasets

The configured `FeatureHasher` is applied to the chronological training, validation, and test datasets.

This transformation converts the selected numerical and categorical predictors into a fixed-size sparse vector stored in the `features` column required by Spark machine learning estimators.

The transformed DataFrames retain `FL_DATE` and the target variable so they can be used for chronological hyperparameter tuning and final model evaluation.


In [0]:
for dataframe_name, dataframe in {
    "df_train_hashed": df_train_hashed,
    "df_validation_hashed": df_validation_hashed,
    "df_test_hashed": df_test_hashed,
}.items():
    missing_columns = sorted(
        set(cfg.MODEL_HASHED_OUTPUT_COLUMNS) - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing required columns: "
            f"{missing_columns}"
        )

    print(
        f"{dataframe_name} ready for modeling with "
        f"{dataframe.count():,} rows."
    )


#### Fit and apply the preprocessing pipeline

The preprocessing pipeline is fitted exclusively on the training dataset.

The fitted pipeline is then applied unchanged to the training, validation, and test datasets. This ensures that category indexing and one-hot encoding are learned only from the training period.

The resulting `features` column contains the assembled numerical and encoded categorical predictors required by Spark ML models.


In [0]:
print("Feature hashing applied successfully.")
print(f"Training rows: {df_train_hashed.count():,}")
print(f"Validation rows: {df_validation_hashed.count():,}")
print(f"Test rows: {df_test_hashed.count():,}")

display(
    df_train_hashed
    .select(
        *cfg.MODELING_JOIN_KEY_COLUMNS,
        "features",
        cfg.TARGET_COLUMN,
    )
    .limit(10)
)


#### Train multiple candidate models

Multiple classification algorithms will be trained and compared using the same chronologically separated datasets.

The first candidate is Logistic Regression, which serves as the baseline model. It provides a computationally efficient benchmark and estimates the probability that a scheduled flight will arrive at least 15 minutes late.

The model is trained using the training period only. Validation data will be used later to evaluate performance and guide model selection.


#### Majority Class Baseline

Before training machine learning models, a simple majority-class baseline is evaluated.

The majority-class classifier predicts every flight as the most frequent class observed in the training data. In this dataset, the majority class is **on-time arrival (ARR_DEL15 = 0)**.

Although this baseline is intentionally simple, it provides an important reference point for determining whether more sophisticated machine learning models deliver meaningful predictive improvements.


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

# ---------------------------------------------------
# Majority class prediction
# ---------------------------------------------------

majority_class = (
    df_train_prepared
    .groupBy(TARGET_COLUMN)
    .count()
    .orderBy(F.desc("count"))
    .first()[TARGET_COLUMN]
)

baseline_predictions = (
    df_validation_prepared
    .withColumn(
        "prediction",
        F.lit(float(majority_class))
    )
    .withColumn(
        "rawPrediction",
        F.array(
            F.lit(1.0),
            F.lit(0.0)
        )
    )
    .withColumn(
        "probability",
        F.array(
            F.lit(1.0),
            F.lit(0.0)
        )
    )
)

print(f"Majority class: {majority_class}")

#### Majority-Class Baseline Result

The training dataset's majority class is `ARR_DEL15 = 0`, representing flights that arrived less than 15 minutes late.

Therefore, the majority-class baseline predicts every validation record as on time. This provides a simple lower-bound benchmark for evaluating whether the machine-learning models produce meaningful improvement.


#### Evaluate the Majority-Class Baseline

The majority-class baseline is evaluated on the validation dataset using the same performance metrics that will be applied to all candidate machine learning models.

Although this classifier predicts every flight as the majority class (on-time arrival), it provides an important benchmark for determining whether more sophisticated models deliver meaningful predictive improvements.

The following metrics are reported:

- Accuracy
- Weighted Precision
- Weighted Recall
- Weighted F1-score

ROC AUC and PR AUC are not applicable because the majority-class baseline does not produce meaningful probability estimates.


In [0]:
baseline_accuracy = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="accuracy",
).evaluate(baseline_predictions)

baseline_precision = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedPrecision",
).evaluate(baseline_predictions)

baseline_recall = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedRecall",
).evaluate(baseline_predictions)

baseline_f1 = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="f1",
).evaluate(baseline_predictions)

majority_baseline_results = spark.createDataFrame(
    [
        (
            "Majority Class Baseline",
            round(baseline_accuracy, 4),
            round(baseline_precision, 4),
            round(baseline_recall, 4),
            round(baseline_f1, 4),
        )
    ],
    [
        "MODEL",
        "ACCURACY",
        "WEIGHTED_PRECISION",
        "WEIGHTED_RECALL",
        "WEIGHTED_F1_SCORE",
    ],
)

display(majority_baseline_results)

#### Majority-Class Baseline Interpretation

The majority-class baseline achieved an overall accuracy of **81.45%**, slightly exceeding the Logistic Regression baseline.

However, this classifier predicts every flight as the majority class (on-time arrival) and therefore cannot identify delayed flights. Consequently, the baseline provides a useful lower-bound benchmark but is not suitable for operational decision support.

The Logistic Regression model is expected to outperform the majority-class baseline on delay-specific evaluation metrics despite having a similar overall accuracy.


#### Majority-Class Baseline Confusion Matrix

A confusion matrix is generated to evaluate the prediction behavior of the majority-class baseline.

Since this classifier predicts every flight as the majority class (on-time arrival), the confusion matrix illustrates its inability to identify delayed flights despite achieving relatively high overall accuracy.

The confusion matrix reports:

- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)
- True Positives (TP)

This analysis provides a reference point for comparing more sophisticated machine learning models.


In [0]:
baseline_confusion = (
    baseline_predictions
    .groupBy(
        F.col(TARGET_COLUMN).alias("ACTUAL"),
        F.col("prediction").cast("int").alias("PREDICTED"),
    )
    .count()
    .orderBy(
        "ACTUAL",
        "PREDICTED",
    )
)

display(baseline_confusion)

#### Majority-Class Baseline Confusion Matrix Interpretation

The majority-class baseline predicts every validation observation as the majority class (`ARR_DEL15 = 0`).

Consequently:

- All on-time flights are predicted correctly.
- No delayed flights are identified.
- The model produces no false-positive predictions because it never predicts the delayed class.
- Every delayed flight becomes a false negative.

Although this classifier achieves relatively high overall accuracy due to the class imbalance, it has no operational value because it cannot identify flights at risk of delay.


#### Logistic Regression Baseline

A binary Logistic Regression model is trained using the assembled `features` vector and the `ARR_DEL15` target.

The initial model uses moderate regularization and a limited number of iterations to establish a baseline. Hyperparameter tuning will be performed later after the candidate models have been compared.


In [0]:
from pyspark.ml.classification import LogisticRegression


logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=cfg.SELECTED_LR_MAX_ITER,
    regParam=0.01,
    elasticNetParam=cfg.SELECTED_LR_PARAMS["elasticNetParam"],
    standardization=True,
    family="binomial",
)

logistic_model = logistic_regression.fit(
    df_train_prepared.select(
        "features",
        TARGET_COLUMN,
    )
)

print("Logistic Regression baseline trained successfully.")
print(f"Iterations completed: {logistic_model.summary.totalIterations}")
print(f"Intercept: {logistic_model.intercept:.6f}")
print(f"Coefficient vector size: {logistic_model.coefficients.size}")

#### Evaluate the Logistic Regression Baseline

The Logistic Regression model is first evaluated on the validation dataset before additional candidate models are trained.

The validation dataset was not used during model fitting, making it suitable for an unbiased assessment of predictive performance.

The following performance metrics are reported:

- Accuracy
- Precision
- Recall
- F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)

These metrics establish the baseline against which subsequent models will be compared.


In [0]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

# ----------------------------------------------------
# Generate validation predictions
# ----------------------------------------------------

validation_predictions = logistic_model.transform(
    df_validation_prepared
)

# ----------------------------------------------------
# Classification metrics
# ----------------------------------------------------

accuracy = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="accuracy",
).evaluate(validation_predictions)

precision = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedPrecision",
).evaluate(validation_predictions)

recall = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedRecall",
).evaluate(validation_predictions)

f1_score = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="f1",
).evaluate(validation_predictions)

roc_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(validation_predictions)

pr_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
).evaluate(validation_predictions)

# ----------------------------------------------------
# Display results
# ----------------------------------------------------

baseline_results = spark.createDataFrame(
    [
        (
            "Logistic Regression",
            round(accuracy, 4),
            round(precision, 4),
            round(recall, 4),
            round(f1_score, 4),
            round(roc_auc, 4),
            round(pr_auc, 4),
        )
    ],
    [
        "MODEL",
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
    ],
)

display(baseline_results)

#### Logistic Regression Confusion Matrix

A confusion matrix is generated to examine the classification outcomes of the Logistic Regression baseline.

The confusion matrix summarizes:

- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)
- True Positives (TP)

This provides additional insight into the types of prediction errors made by the model beyond the aggregate evaluation metrics.


In [0]:
confusion_matrix = (
    validation_predictions
    .groupBy(
        F.col(TARGET_COLUMN).alias("ACTUAL"),
        F.col("prediction").cast("int").alias("PREDICTED"),
    )
    .count()
    .orderBy(
        "ACTUAL",
        "PREDICTED",
    )
)

display(confusion_matrix)

#### Confusion Matrix Interpretation

The Logistic Regression confusion matrix is interpreted as follows:

- **True Negatives (TN):** Correctly predicted on-time flights.
- **False Positives (FP):** Flights predicted as delayed that actually arrived on time.
- **False Negatives (FN):** Flights predicted as on time that were actually delayed.
- **True Positives (TP):** Correctly predicted delayed flights.

This analysis provides insight into the model's prediction errors beyond overall accuracy.


In [0]:
TN = confusion_matrix.filter(
    (F.col("ACTUAL") == 0) &
    (F.col("PREDICTED") == 0)
).first()["count"]

FP = confusion_matrix.filter(
    (F.col("ACTUAL") == 0) &
    (F.col("PREDICTED") == 1)
).first()["count"]

FN = confusion_matrix.filter(
    (F.col("ACTUAL") == 1) &
    (F.col("PREDICTED") == 0)
).first()["count"]

TP = confusion_matrix.filter(
    (F.col("ACTUAL") == 1) &
    (F.col("PREDICTED") == 1)
).first()["count"]

confusion_summary = spark.createDataFrame(
    [
        ("True Negative (TN)", TN),
        ("False Positive (FP)", FP),
        ("False Negative (FN)", FN),
        ("True Positive (TP)", TP),
    ],
    ["CLASSIFICATION_RESULT", "COUNT"],
)

display(confusion_summary)

#### Logistic Regression Confusion-Matrix Interpretation

The Logistic Regression baseline correctly classified most on-time flights, producing 939,046 true negatives and only 5,742 false positives.

However, the model detected only 3,382 delayed flights while incorrectly classifying 211,728 delayed flights as on time. This corresponds to a delayed-flight recall of approximately 1.57%.

Therefore, the baseline's high accuracy is largely driven by strong performance on the majority on-time class. Additional candidate models, class weighting, and decision-threshold tuning are necessary to improve delayed-flight detection.


#### Model Performance Comparison

To support objective model selection, the performance of every candidate model is summarized in a single comparison table.

Each model is evaluated using the same validation dataset and the same evaluation metrics.

The comparison includes:

- Accuracy
- Precision
- Recall
- F1-score
- ROC AUC
- Precision–Recall AUC

This table will be updated as additional candidate models are trained and evaluated.


In [0]:
model_comparison = spark.createDataFrame(
    [
        (
            "Majority Class Baseline",
            baseline_accuracy,
            baseline_precision,
            baseline_recall,
            baseline_f1,
            None,
            None,
        ),
        (
            "Logistic Regression",
            accuracy,
            precision,
            recall,
            f1_score,
            roc_auc,
            pr_auc,
        ),
    ],
    schema="""
        MODEL string,
        ACCURACY double,
        PRECISION double,
        RECALL double,
        F1_SCORE double,
        ROC_AUC double,
        PR_AUC double
    """,
)

display(
    model_comparison
    .orderBy("MODEL")
)

#### Comparison-Table Note

ROC AUC and Precision–Recall AUC are not reported for the majority-class baseline because it assigns the same prediction to every record and does not generate meaningful risk rankings.

The null values are therefore expected and do not indicate a data-processing error.


#### Random Forest Classifier

The second candidate model is a Random Forest classifier.

Random Forest can capture nonlinear relationships and interactions among predictors that may not be represented adequately by Logistic Regression.

Because ensemble-tree training on the complete 4.59-million-record dataset is computationally expensive in the available Serverless environment, the model is trained using a reproducible stratified sample of the training period. Stratified sampling preserves both on-time and delayed-flight observations while maintaining training-only data use.

The fitted model will still be evaluated on the complete validation dataset.


In [0]:
# -------------------------------------------------------
# Reproducible stratified sample for tree-based models
# -------------------------------------------------------

TREE_SAMPLE_FRACTIONS = cfg.TREE_TUNING_SAMPLE_FRACTIONS

df_train_tree = (
    df_train_prepared
    .sampleBy(
        col=TARGET_COLUMN,
        fractions=TREE_SAMPLE_FRACTIONS,
        seed=cfg.RANDOM_SEED,
    )
    .select(
        "features",
        TARGET_COLUMN,
    )
)

tree_sample_summary = (
    df_train_tree
    .groupBy(TARGET_COLUMN)
    .count()
    .orderBy(TARGET_COLUMN)
)

tree_sample_count = df_train_tree.count()

print(f"Tree-model training sample rows: {tree_sample_count:,}")
display(tree_sample_summary)

#### Stratified Training Sample Interpretation

The stratified sampling procedure retained approximately 670 thousand training records while increasing the representation of delayed flights.

Compared with the original training dataset, the sampled dataset contains a substantially more balanced class distribution, allowing tree-based models to learn delay-related patterns more effectively while reducing computational cost.

Only the training dataset was sampled. The validation and test datasets remain unchanged to preserve an unbiased evaluation of model performance.


In [0]:
from pyspark.ml.classification import RandomForestClassifier

random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",

    # Conservative baseline parameters
    numTrees=5,
    maxDepth=6,
    maxBins=32,
    minInstancesPerNode=20,

    seed=cfg.RANDOM_SEED,
)

random_forest_model = random_forest.fit(
    df_train_prepared.select(
        "features",
        TARGET_COLUMN,
    )
)

print("Random Forest trained successfully.")
print(f"Trees: {random_forest_model.getNumTrees}")

#### Evaluate the Random Forest Baseline

The Random Forest classifier is evaluated using the validation dataset.

The validation dataset was not used during model training and therefore provides an unbiased estimate of predictive performance.

The following evaluation metrics are reported:

- Accuracy
- Precision
- Recall
- F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)

These metrics are compared directly with the Majority-Class Baseline and Logistic Regression models.


In [0]:
rf_validation_predictions = random_forest_model.transform(
    df_validation_prepared
)

rf_accuracy = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="accuracy",
).evaluate(rf_validation_predictions)

rf_precision = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedPrecision",
).evaluate(rf_validation_predictions)

rf_recall = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedRecall",
).evaluate(rf_validation_predictions)

rf_f1 = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="f1",
).evaluate(rf_validation_predictions)

rf_roc_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(rf_validation_predictions)

rf_pr_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
).evaluate(rf_validation_predictions)

rf_results = spark.createDataFrame(
    [
        (
            "Random Forest",
            round(rf_accuracy, 4),
            round(rf_precision, 4),
            round(rf_recall, 4),
            round(rf_f1, 4),
            round(rf_roc_auc, 4),
            round(rf_pr_auc, 4),
        )
    ],
    [
        "MODEL",
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
    ],
)

display(rf_results)

#### Random Forest Confusion Matrix

A confusion matrix is generated to evaluate the classification behavior of the baseline Random Forest model.

The confusion matrix summarizes:

- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)
- True Positives (TP)

These results provide insight into the types of prediction errors made by the Random Forest model and serve as the baseline before hyperparameter tuning.


In [0]:
rf_confusion_matrix = (
    rf_validation_predictions
    .groupBy(
        F.col(TARGET_COLUMN).alias("ACTUAL"),
        F.col("prediction").cast("int").alias("PREDICTED"),
    )
    .count()
    .orderBy(
        "ACTUAL",
        "PREDICTED",
    )
)

display(rf_confusion_matrix)

#### Random Forest Baseline Interpretation

The baseline Random Forest classifier predicted every validation observation as the majority class (on-time arrival), producing the same confusion matrix as the Majority-Class Baseline.

Although the model completed training successfully, the selected baseline hyperparameters were too conservative to identify delayed flights.

This outcome establishes a meaningful baseline and motivates the hyperparameter-tuning stage, where model complexity will be increased to improve delayed-flight detection while maintaining generalization performance.


#### Updated Model Performance Comparison

The comparison table is updated after completing the baseline models.

At this stage, three reference models have been evaluated:

- Majority-Class Baseline
- Logistic Regression
- Baseline Random Forest

The Random Forest results presented below correspond to the initial baseline configuration prior to hyperparameter tuning. The comparison table will be updated again after tuning to reflect the best-performing Random Forest model.


In [0]:
model_comparison = spark.createDataFrame(
    [
        (
            "Majority Class Baseline",
            round(baseline_accuracy, 4),
            round(baseline_precision, 4),
            round(baseline_recall, 4),
            round(baseline_f1, 4),
            None,
            None,
        ),
        (
            "Logistic Regression",
            round(accuracy, 4),
            round(precision, 4),
            round(recall, 4),
            round(f1_score, 4),
            round(roc_auc, 4),
            round(pr_auc, 4),
        ),
        (
            "Random Forest (Baseline)",
            round(rf_accuracy, 4),
            round(rf_precision, 4),
            round(rf_recall, 4),
            round(rf_f1, 4),
            round(rf_roc_auc, 4),
            round(rf_pr_auc, 4),
        ),
    ],
    schema="""
        MODEL string,
        ACCURACY double,
        PRECISION double,
        RECALL double,
        F1_SCORE double,
        ROC_AUC double,
        PR_AUC double
    """
)

display(
    model_comparison.orderBy("MODEL")
)

#### Comparison Interpretation

The comparison of the baseline models demonstrates that overall accuracy alone is not an appropriate metric for evaluating flight-delay prediction because the dataset is dominated by on-time flights.

The Majority-Class Baseline achieved the highest overall accuracy by predicting every flight as on time, but it was unable to identify delayed flights and therefore provides little operational value.

The Logistic Regression model produced a slightly lower overall accuracy while achieving the highest ROC AUC, Precision–Recall AUC, and F1-score among the evaluated baseline models. These results indicate that Logistic Regression provides better discrimination between delayed and on-time flights despite the class imbalance.

The baseline Random Forest model achieved performance similar to the Majority-Class Baseline, suggesting that the initial hyperparameter configuration was too conservative and unable to learn meaningful delay patterns.

These results justify the next phase of the workflow, where Random Forest hyperparameters will be optimized to improve delayed-flight detection before comparing the model with a gradient-boosting algorithm.


#### XGBoost Baseline

The third candidate model is standard XGBoost, implemented using `xgboost.XGBClassifier`.

Databricks Free Edition provides Serverless compute, which does not support the direct SparkContext access required by `xgboost.spark.SparkXGBClassifier`. Standard XGBoost is therefore trained locally using SciPy compressed sparse row (CSR) matrices created from the existing Spark feature vectors. This preserves the sparse hashed-feature representation without converting it into a dense pandas dataset.

To remain within the memory and runtime limits of Free Edition, the XGBoost baseline uses a reproducible class-stratified training sample capped at approximately **50,000 records**. Class-stratified sampling improves representation of delayed flights while ensuring that only observations from the chronological training period are used.

The baseline validation evaluation uses a reproducible uniform sample capped at approximately **50,000 records**. Uniform rather than class-stratified validation sampling is used to preserve the natural class distribution of the validation period.

The same approximate 50,000-record validation limit and random seed are subsequently applied to Logistic Regression, Random Forest, and XGBoost during chronological hyperparameter tuning. This ensures that the tuned candidate models are evaluated using consistent validation sample sizes and reproducible sampling rules.

The baseline model is evaluated using Accuracy, weighted Precision, weighted Recall, weighted F1-score, ROC AUC, PR AUC, and its confusion matrix. Final candidate selection is based on the average chronological tuning results rather than the single baseline evaluation.


In [0]:
try:
    import numpy as np

    from scipy.sparse import csr_matrix

    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        precision_recall_fscore_support,
        roc_auc_score,
    )

    from xgboost import XGBClassifier

except ImportError as error:
    raise ImportError(
        "Standard XGBoost and its local dependencies are required. "
        "Run `%pip install xgboost==2.1.4 scipy scikit-learn`, "
        "restart Python once, and rerun this notebook from "
        "the beginning."
    ) from error


# ============================================================
# 1. Driver-memory safeguards for Databricks Free Edition
# ============================================================

# Maximum local XGBoost training rows per chronological fold.
XGB_MAX_TRAIN_ROWS = 50_000

# Maximum baseline XGBoost validation rows.
XGB_MAX_VALIDATION_ROWS = 50_000

# Shared tuning-validation limit for LR, RF, and XGBoost.
TUNING_VALIDATION_MAX_ROWS = 50_000

# Retained for other batch-processing utilities if needed.
XGB_VALIDATION_BATCH_SIZE = 10_000


# ============================================================
# 2. Reproducible bounded Spark sampling
# ============================================================

def bounded_spark_sample(
    dataframe,
    maximum_rows,
    *,
    stratified=False,
):
    """
    Return a reproducible, driver-safe Spark sample.

    Stratified sampling is used for XGBoost training data.
    Uniform sampling is used for the baseline validation sample.
    """
    row_count = dataframe.count()

    if row_count == 0:
        raise ValueError(
            "The DataFrame provided for sampling contains zero rows."
        )

    if row_count <= maximum_rows:
        return dataframe

    if stratified:
        class_counts = {
            row[TARGET_COLUMN]: row["count"]
            for row in (
                dataframe
                .groupBy(TARGET_COLUMN)
                .count()
                .collect()
            )
        }

        target_per_class = (
            maximum_rows
            / max(len(class_counts), 1)
        )

        fractions = {
            label: min(
                1.0,
                target_per_class / count,
            )
            for label, count in class_counts.items()
        }

        return dataframe.sampleBy(
            TARGET_COLUMN,
            fractions=fractions,
            seed=cfg.RANDOM_SEED,
        )

    return dataframe.sample(
        withReplacement=False,
        fraction=min(
            1.0,
            maximum_rows / row_count,
        ),
        seed=cfg.RANDOM_SEED,
    )


# ============================================================
# 3. Convert a Spark vector DataFrame to SciPy CSR
# ============================================================

def spark_vectors_to_csr(dataframe):
    """
    Stream Spark vectors into a SciPy CSR matrix.

    This function is intended for bounded training or baseline
    validation samples, not complete validation folds.
    """
    data = []
    indices = []
    indptr = [0]
    labels = []

    feature_count = None

    row_iterator = (
        dataframe
        .select(
            "features",
            TARGET_COLUMN,
        )
        .toLocalIterator()
    )

    for row in row_iterator:
        vector = row["features"]
        feature_count = vector.size

        if hasattr(vector, "indices"):
            indices.extend(
                int(index)
                for index in vector.indices
            )

            data.extend(
                float(value)
                for value in vector.values
            )

        else:
            dense_values = np.asarray(
                vector,
                dtype=np.float32,
            )

            nonzero = np.flatnonzero(
                dense_values
            )

            indices.extend(
                nonzero.tolist()
            )

            data.extend(
                dense_values[nonzero].tolist()
            )

        indptr.append(
            len(data)
        )

        labels.append(
            int(row[TARGET_COLUMN])
        )

    if not labels or feature_count is None:
        raise ValueError(
            "The local XGBoost sample contains zero rows."
        )

    matrix = csr_matrix(
        (
            np.asarray(
                data,
                dtype=np.float32,
            ),
            np.asarray(
                indices,
                dtype=np.int32,
            ),
            np.asarray(
                indptr,
                dtype=np.int64,
            ),
        ),
        shape=(
            len(labels),
            feature_count,
        ),
        dtype=np.float32,
    )

    label_array = np.asarray(
        labels,
        dtype=np.int8,
    )

    return matrix, label_array


# ============================================================
# 4. Calculate classification metrics
# ============================================================

def calculate_xgboost_metrics(
    labels,
    probabilities,
):
    """
    Calculate metrics matching the Spark tuning-result schema.
    """
    predictions = (
        probabilities >= 0.5
    ).astype(np.int8)

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    (
        delay_precision,
        delay_recall,
        delay_f1,
        _,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0,
    )

    unique_labels = np.unique(
        labels
    )

    if len(unique_labels) < 2:
        raise ValueError(
            "XGBoost validation data must contain "
            "both target classes."
        )

    metrics = {
        "ACCURACY": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "PRECISION": float(
            weighted_precision
        ),
        "RECALL": float(
            weighted_recall
        ),
        "F1_SCORE": float(
            weighted_f1
        ),
        "ROC_AUC": float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        "PR_AUC": float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        "DELAY_PRECISION": float(
            delay_precision
        ),
        "DELAY_RECALL": float(
            delay_recall
        ),
        "DELAY_F1": float(
            delay_f1
        ),
    }

    return (
        metrics,
        predictions,
        probabilities,
    )


# ============================================================
# 5. Evaluate a bounded local XGBoost dataset
# ============================================================

def evaluate_local_xgboost(
    model,
    features,
    labels,
):
    """
    Evaluate an XGBoost model using an existing local CSR matrix.
    """
    probabilities = model.predict_proba(
        features
    )[:, 1]

    return calculate_xgboost_metrics(
        labels,
        probabilities,
    )


# ============================================================
# 6. Evaluate a complete Spark validation fold in batches
# ============================================================

def evaluate_complete_xgboost_validation(
    model,
    validation_df,
    batch_size=XGB_VALIDATION_BATCH_SIZE,
):
    """
    Score every validation row in driver-safe batches.

    Only one sparse feature batch is retained at a time.
    Labels and probabilities are retained for final metrics.
    """
    all_labels = []
    all_probabilities = []

    batch_rows = []

    def score_batch(rows):
        data = []
        indices = []
        indptr = [0]
        labels = []

        feature_count = None

        for row in rows:
            vector = row["features"]
            feature_count = vector.size

            if hasattr(vector, "indices"):
                indices.extend(
                    int(index)
                    for index in vector.indices
                )

                data.extend(
                    float(value)
                    for value in vector.values
                )

            else:
                dense_values = np.asarray(
                    vector,
                    dtype=np.float32,
                )

                nonzero = np.flatnonzero(
                    dense_values
                )

                indices.extend(
                    nonzero.tolist()
                )

                data.extend(
                    dense_values[nonzero].tolist()
                )

            indptr.append(
                len(data)
            )

            labels.append(
                int(row[TARGET_COLUMN])
            )

        if not labels or feature_count is None:
            raise ValueError(
                "An XGBoost validation batch "
                "contains zero rows."
            )

        matrix = csr_matrix(
            (
                np.asarray(
                    data,
                    dtype=np.float32,
                ),
                np.asarray(
                    indices,
                    dtype=np.int32,
                ),
                np.asarray(
                    indptr,
                    dtype=np.int64,
                ),
            ),
            shape=(
                len(labels),
                feature_count,
            ),
            dtype=np.float32,
        )

        label_array = np.asarray(
            labels,
            dtype=np.int8,
        )

        probability_array = (
            model.predict_proba(
                matrix
            )[:, 1]
        )

        return (
            label_array,
            probability_array,
        )

    validation_iterator = (
        validation_df
        .select(
            "features",
            TARGET_COLUMN,
        )
        .toLocalIterator()
    )

    for row in validation_iterator:
        batch_rows.append(row)

        if len(batch_rows) >= batch_size:
            (
                batch_labels,
                batch_probabilities,
            ) = score_batch(
                batch_rows
            )

            all_labels.append(
                batch_labels
            )

            all_probabilities.append(
                batch_probabilities
            )

            batch_rows.clear()

    if batch_rows:
        (
            batch_labels,
            batch_probabilities,
        ) = score_batch(
            batch_rows
        )

        all_labels.append(
            batch_labels
        )

        all_probabilities.append(
            batch_probabilities
        )

        batch_rows.clear()

    if not all_labels:
        raise ValueError(
            "The complete XGBoost validation fold "
            "contains zero rows."
        )

    labels = np.concatenate(
        all_labels
    )

    probabilities = np.concatenate(
        all_probabilities
    )

    return calculate_xgboost_metrics(
        labels,
        probabilities,
    )


print("Standard XGBoost is available.")
print(f"Estimator: {XGBClassifier.__name__}")
print(
    f"Maximum local training rows: "
    f"{XGB_MAX_TRAIN_ROWS:,}"
)
print(
    f"Baseline validation-row limit: "
    f"{XGB_MAX_VALIDATION_ROWS:,}"
)
print(
    f"Complete-fold scoring batch size: "
    f"{XGB_VALIDATION_BATCH_SIZE:,}"
)

print(
    f"Tuning validation-row limit: "
    f"{TUNING_VALIDATION_MAX_ROWS:,}"
)

#### Train the XGBoost Baseline

The baseline XGBoost classifier is implemented using the standard `xgboost.XGBClassifier`.

Training uses a reproducible, class-stratified sample drawn exclusively from the chronological training period. The sample is capped at approximately **50,000 records** to remain within the driver-memory and runtime limits of Databricks Free Edition.

The existing Spark feature vectors are converted directly into a SciPy compressed sparse row (CSR) matrix. This preserves the sparse hashed-feature representation and avoids creating a potentially memory-intensive dense pandas dataset.

The baseline configuration uses:

- 50 boosting trees
- Maximum tree depth of 4
- Learning rate of 0.10
- Row-subsampling rate of 0.80
- Feature-subsampling rate of 0.80
- Histogram-based tree construction

The baseline establishes an initial XGBoost performance reference before chronological hyperparameter tuning. Its validation data remains separate from model training and is evaluated in the following section.

In [0]:
xgb_baseline_training_df = bounded_spark_sample(
    df_train_tree.select("features", TARGET_COLUMN),
    XGB_MAX_TRAIN_ROWS,
    stratified=True,
)

X_xgb_train, y_xgb_train = spark_vectors_to_csr(
    xgb_baseline_training_df
)

xgboost_model = XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.10,
    subsample=0.80,
    colsample_bytree=0.80,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=cfg.RANDOM_SEED,
    n_jobs=-1,
)

xgboost_model.fit(X_xgb_train, y_xgb_train)

print("XGBoost baseline trained successfully.")
print(f"Local training rows: {X_xgb_train.shape[0]:,}")
print(f"Features: {X_xgb_train.shape[1]:,}")
print("Trees: 50")
print("Maximum depth: 4")
print("Learning rate: 0.10")


#### Evaluate the XGBoost Baseline

The trained XGBoost baseline is evaluated on a reproducible uniform sample from the complete chronological validation period.
The sample is not class-balanced, so it retains the period's natural class distribution. The same overall and ranking metrics used for the Spark candidates are calculated locally.

In [0]:
xgb_baseline_validation_df = bounded_spark_sample(
    df_validation_prepared.select("features", TARGET_COLUMN),
    XGB_MAX_VALIDATION_ROWS,
    stratified=False,
)

X_xgb_validation, y_xgb_validation = spark_vectors_to_csr(
    xgb_baseline_validation_df
)

xgb_baseline_metrics, xgb_predictions, xgb_probabilities = (
    evaluate_local_xgboost(
        xgboost_model,
        X_xgb_validation,
        y_xgb_validation,
    )
)

xgb_accuracy = xgb_baseline_metrics["ACCURACY"]
xgb_precision = xgb_baseline_metrics["PRECISION"]
xgb_recall = xgb_baseline_metrics["RECALL"]
xgb_f1 = xgb_baseline_metrics["F1_SCORE"]
xgb_roc_auc = xgb_baseline_metrics["ROC_AUC"]
xgb_pr_auc = xgb_baseline_metrics["PR_AUC"]

xgb_results = spark.createDataFrame(
    [
        (
            "XGBoost",
            round(xgb_accuracy, 4),
            round(xgb_precision, 4),
            round(xgb_recall, 4),
            round(xgb_f1, 4),
            round(xgb_roc_auc, 4),
            round(xgb_pr_auc, 4),
        )
    ],
    [
        "MODEL",
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
    ],
)

display(xgb_results)


#### XGBoost Confusion Matrix

The confusion matrix summarizes XGBoost predictions on the reproducible validation sample. Its counts refer to that sample rather than the complete validation period.


In [0]:
xgb_confusion_matrix = spark.createDataFrame(
    [
        (int(actual), int(predicted))
        for actual, predicted in zip(
            y_xgb_validation.tolist(),
            xgb_predictions.tolist(),
        )
    ],
    ["ACTUAL", "PREDICTED"],
).groupBy(
    "ACTUAL",
    "PREDICTED",
).count().orderBy(
    "ACTUAL",
    "PREDICTED",
)

display(xgb_confusion_matrix)


#### XGBoost Confusion-Matrix Interpretation

The XGBoost baseline was evaluated using a reproducible uniform validation sample of **50,010 flights**. The confusion matrix produced:

- **26,054 true negatives:** on-time flights correctly classified as on time.
- **14,849 false positives:** on-time flights incorrectly classified as delayed.
- **3,605 false negatives:** delayed flights incorrectly classified as on time.
- **5,502 true positives:** delayed flights correctly identified.

The baseline identified approximately **60.42% of the delayed flights**, corresponding to delayed-flight Recall of approximately **0.6042**. Its delayed-flight Precision was approximately **0.2704**, meaning that about 27.04% of the flights classified as delayed were actually delayed. The resulting delayed-flight F1-score was approximately **0.3736**.

The model achieved overall Accuracy of approximately **63.10%**. Its relatively high delayed-flight Recall demonstrates that the baseline can identify a meaningful proportion of delayed flights. However, this capability comes with a substantial number of false-positive alerts, indicating that many flights flagged for operational attention would ultimately arrive on time.

This trade-off is relevant to the project’s operational objective. Missing a delayed flight may prevent proactive intervention, while a false-positive alert may consume limited operational resources unnecessarily. Delayed-flight Recall must therefore be considered together with Precision, F1-score, PR AUC, and the available intervention capacity.

Because these results were calculated from a reproducible validation sample rather than the complete validation period, they should be treated as baseline estimates. Final model selection should rely on the average chronological tuning results, where Logistic Regression, Random Forest, and XGBoost are evaluated using the same validation limit and sampling procedure.


#### Updated Baseline Model Comparison

The comparison table is updated after evaluating all required baseline models.

At this stage, the following candidate models have been completed:

- Majority-Class Baseline
- Logistic Regression
- Random Forest (Baseline)
- XGBoost (Baseline)

The tree-based models shown below correspond to their initial baseline configurations prior to hyperparameter tuning. The comparison table will be updated after tuning to reflect the best-performing models.


In [0]:
model_comparison = spark.createDataFrame(
    [
        (
            "Majority Class Baseline",
            round(baseline_accuracy, 4),
            round(baseline_precision, 4),
            round(baseline_recall, 4),
            round(baseline_f1, 4),
            None,
            None,
        ),
        (
            "Logistic Regression",
            round(accuracy, 4),
            round(precision, 4),
            round(recall, 4),
            round(f1_score, 4),
            round(roc_auc, 4),
            round(pr_auc, 4),
        ),
        (
            "Random Forest (Baseline)",
            round(rf_accuracy, 4),
            round(rf_precision, 4),
            round(rf_recall, 4),
            round(rf_f1, 4),
            round(rf_roc_auc, 4),
            round(rf_pr_auc, 4),
        ),
        (
            "XGBoost (Baseline)",
            round(xgb_accuracy, 4),
            round(xgb_precision, 4),
            round(xgb_recall, 4),
            round(xgb_f1, 4),
            round(xgb_roc_auc, 4),
            round(xgb_pr_auc, 4),
        ),
    ],
    schema="""
        MODEL string,
        ACCURACY double,
        PRECISION double,
        RECALL double,
        F1_SCORE double,
        ROC_AUC double,
        PR_AUC double
    """,
)

display(model_comparison.orderBy("MODEL"))


#### Baseline Model Comparison Interpretation

The baseline comparison confirms that overall Accuracy and weighted metrics alone are insufficient for evaluating flight-delay prediction because the dataset is dominated by on-time flights.

The Majority-Class Baseline and baseline Random Forest achieved the highest Accuracy of **0.8145**. However, both models effectively predicted every flight as on time and therefore failed to identify delayed flights. Their high Accuracy reflects the target-class imbalance rather than meaningful operational performance. The Majority-Class Baseline does not produce useful probability rankings, so ROC AUC and PR AUC are not reported.

Logistic Regression achieved Accuracy of **0.8125** and the highest weighted F1-score of **0.7355**. Its ROC AUC was **0.6518**, while its PR AUC was **0.2794**. Although these metrics demonstrate better discrimination than the Majority-Class Baseline and Random Forest, its baseline confusion matrix showed that the default classification threshold identified only a small proportion of delayed flights.

XGBoost achieved lower Accuracy of **0.6310** and weighted F1-score of **0.6720**, reflecting its greater number of false-positive delay alerts. However, it achieved the highest weighted Precision (**0.7677**), ROC AUC (**0.6595**), and PR AUC (**0.2808**) among the baseline candidates. Its confusion matrix also showed that it identified approximately **60.42% of delayed flights**, demonstrating substantially stronger delayed-flight detection at the default threshold.

The XGBoost results illustrate an important operational trade-off. The model detects considerably more delayed flights but also incorrectly flags many on-time flights. This behavior lowers overall Accuracy while potentially providing greater value when the cost of missing a delayed flight is higher than the cost of reviewing an unnecessary alert.

The XGBoost baseline was evaluated using a reproducible validation sample of approximately 50,000 records, whereas the Spark baseline models were evaluated using the complete validation dataset. Consequently, small differences in baseline metrics should not be interpreted as definitive evidence that one model is superior.

The chronological tuning stage provides the fairer candidate comparison. Logistic Regression, Random Forest, and XGBoost are evaluated using the same approximate 50,000-record validation limit, chronological fold boundaries, and reproducible sampling procedure. Final model selection should therefore rely on the tuned results rather than this baseline table.


#### Hyperparameter Tuning

The baseline models established the initial predictive performance of each candidate algorithm. This phase evaluates multiple hyperparameter configurations while preserving chronological order and preventing temporal data leakage.

Hyperparameter tuning is performed exclusively within the January–August 2025 development period using expanding-window chronological validation folds. Earlier months are used for training, and the immediately following month is used for validation.

The Majority-Class Baseline is excluded because it contains no trainable parameters.

Logistic Regression and Random Forest are tuned using Spark. XGBoost is implemented with the standard `XGBClassifier` because Databricks Free Edition does not support the Spark-native XGBoost estimator. For each XGBoost fold, bounded samples are created only after the chronological training and validation periods have been defined. Training samples are class-stratified, while validation samples are uniformly selected to preserve the natural class distribution.

The tuning process consists of:

1. Establishing the chronological tuning strategy.
2. Creating reusable tuning and evaluation utilities.
3. Tuning Logistic Regression.
4. Tuning Random Forest.
5. Tuning XGBoost.
6. Comparing the strongest configuration from each algorithm.
7. Selecting the candidate final model dynamically for downstream evaluation and explainability.

Candidate configurations are evaluated using overall classification metrics, delayed-flight Precision, delayed-flight Recall, delayed-flight F1-score, ROC AUC, PR AUC, and training time. Particular emphasis is placed on delayed-flight Recall and delayed-flight F1-score because identifying flights at risk of delay is the primary operational objective.


#### Hyperparameter Tuning Strategy

Chronological validation ensures that each candidate model is evaluated using observations from a period later than its corresponding training period.

The January–August 2025 development period is divided into four expanding-window folds:

| Fold | Training period | Validation period |
|---|---|---|
| Fold 1 | January–April 2025 | May 2025 |
| Fold 2 | January–May 2025 | June 2025 |
| Fold 3 | January–June 2025 | July 2025 |
| Fold 4 | January–July 2025 | August 2025 |

Every candidate configuration uses these same chronological date boundaries. Logistic Regression and Random Forest are trained and evaluated through Spark. Standard XGBoost uses bounded, class-stratified training samples and bounded, uniformly selected validation samples because it must run within the driver-memory limitations of Databricks Free Edition.

Sampling occurs only after the chronological training and validation periods have been defined. Consequently, no observation from a validation month is included in its corresponding training sample.

The evaluation considers:

- Overall Accuracy
- Weighted Precision
- Weighted Recall
- Weighted F1-score
- Delayed-flight Precision
- Delayed-flight Recall
- Delayed-flight F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)
- Average training time

For each algorithm, configurations are ranked primarily by delayed-flight Recall, followed by delayed-flight F1-score, PR AUC, ROC AUC, and lower average training time. This ranking reflects the operational objective of identifying as many delayed flights as possible while maintaining useful classification and probability-ranking performance.

The strongest configuration from each algorithm is retained for the tuned candidate-model comparison.


#### Logistic Regression Hyperparameter Tuning

Logistic Regression is first optimized by evaluating different combinations of regularization strength and elastic-net mixing.

These hyperparameters control model complexity and help reduce overfitting while maintaining good generalization performance.

The evaluated parameters include:

- Regularization parameter (`regParam`)
- Elastic-net mixing parameter (`elasticNetParam`)


#### Required Libraries

The required Python libraries for chronological hyperparameter tuning are imported in this section.

These libraries provide functionality for:

Measuring computational training time.

Creating reusable tuning and evaluation functions.

Computing overall and delayed-flight classification metrics.

Computing probability-ranking metrics such as ROC AUC and Precision–Recall AUC.

Working with Spark DataFrames and machine-learning evaluators.

The imported modules are reused throughout the hyperparameter-tuning process for Logistic Regression, Random Forest, and XGBoost.

## Build the Chronological Hyperparameter-Tuning Framework

This section creates a reusable chronological hyperparameter-tuning framework for Logistic Regression and Random Forest. XGBoost follows the same chronological folds and validation limit through a separate local tuning function because it is implemented using the standard XGBClassifier.

The framework uses the prepared January–August 2025 development dataset containing the flight date, target variable, and model-ready feature vector. Four expanding-window validation folds are defined:

Fold

Training period

Validation period

Fold 1

January–April 2025

May 2025

Fold 2

January–May 2025

June 2025

Fold 3

January–June 2025

July 2025

Fold 4

January–July 2025

August 2025

For every fold, the chronological training and validation periods are defined before sampling is performed:

The training fold contains only observations available on or before the specified training cutoff.

Reproducible training sampling is performed only within the chronological training period.

A separate, reproducible, uniformly selected validation sample of approximately 50,000 rows is drawn from the immediately following month.

Uniform validation sampling preserves the month’s natural class distribution rather than artificially balancing the evaluation data.

Every hyperparameter configuration for the same algorithm is evaluated using the same chronological folds and sampling procedure.

Model performance is summarized by averaging the metrics across the four folds.

The validation limit is applied consistently to Logistic Regression, Random Forest, and XGBoost. This provides aligned validation conditions while keeping execution practical within Databricks Free Edition.

Standard XGBoost uses an additional driver-safe training cap of approximately 50,000 rows per fold because it is trained locally using SciPy sparse matrices. Logistic Regression and Random Forest are trained through Spark and use larger training samples. This difference in training exposure is acknowledged when interpreting the tuned-model comparison.

The framework reports overall classification metrics, delayed-flight Precision, delayed-flight Recall, delayed-flight F1-score, ROC AUC, PR AUC, and average training time.

Candidate configurations are ranked using the following order:

Highest delayed-flight Recall.

Highest delayed-flight F1-score.

Highest PR AUC.

Highest ROC AUC.

Lowest average training time.

This design preserves temporal ordering and prevents future observations from leaking into earlier training periods. Sampling is always performed only after the chronological boundaries have been established.


In [0]:
from __future__ import annotations

import time
from typing import Callable

from pyspark.sql import functions as F

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

In [0]:
from __future__ import annotations

import importlib.util
import time

from collections.abc import Callable
from pathlib import Path

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


# ============================================================
# 1. Load project configuration
# ============================================================

_bootstrap = (
    Path.cwd()
    / "notebooks"
    / "import_path.py"
)

if not _bootstrap.is_file():
    _bootstrap = (
        Path.cwd()
        / "import_path.py"
    )

_spec = importlib.util.spec_from_file_location(
    "import_path",
    _bootstrap,
)

_ip = importlib.util.module_from_spec(
    _spec
)

_spec.loader.exec_module(
    _ip
)


# ============================================================
# 2. Global tuning configuration
# ============================================================

TARGET_COLUMN = globals().get(
    "TARGET_COLUMN",
    "ARR_DEL15",
)

RANDOM_SEED = cfg.RANDOM_SEED

REQUIRED_TUNING_COLUMNS = {
    "FL_DATE",
    TARGET_COLUMN,
    "features",
}


# ============================================================
# 3. Locate the prepared training DataFrame
# ============================================================

SOURCE_DATAFRAME_CANDIDATES = [
    "df_train_encoded",
    "df_train_hashed",
    "df_train_features",
    "df_train_model",
    "df_train",
]

source_training_df = None
source_training_df_name = None

for candidate_name in SOURCE_DATAFRAME_CANDIDATES:
    candidate_object = globals().get(
        candidate_name
    )

    if isinstance(
        candidate_object,
        DataFrame,
    ):
        candidate_columns = set(
            candidate_object.columns
        )

        if REQUIRED_TUNING_COLUMNS.issubset(
            candidate_columns
        ):
            source_training_df = (
                candidate_object
            )

            source_training_df_name = (
                candidate_name
            )

            break

if source_training_df is None:
    raise ValueError(
        "A prepared training DataFrame could not be located. "
        "The DataFrame must contain FL_DATE, features, and "
        f"{TARGET_COLUMN}. Add its variable name to "
        "SOURCE_DATAFRAME_CANDIDATES."
    )

print(
    f"Tuning source DataFrame: "
    f"{source_training_df_name}"
)


# ============================================================
# 4. Prepare the complete tuning dataset
# ============================================================

TUNING_TABLE = cfg.TUNING_TABLE

prepared_tuning_df = (
    source_training_df
    .select(
        F.to_date(
            F.col("FL_DATE")
        ).alias("FL_DATE"),

        F.col(
            TARGET_COLUMN
        ).cast(
            "double"
        ).alias(
            TARGET_COLUMN
        ),

        F.col("features"),
    )
    .filter(
        F.col("FL_DATE").isNotNull()
        & F.col(TARGET_COLUMN).isin(
            0.0,
            1.0,
        )
        & F.col("features").isNotNull()
    )
)

(
    prepared_tuning_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        TUNING_TABLE
    )
)

df_tuning_complete = spark.table(
    TUNING_TABLE
)

tuning_row_count = (
    df_tuning_complete.count()
)

if tuning_row_count == 0:
    raise ValueError(
        "The complete tuning dataset contains zero rows. "
        "Verify the source DataFrame and target-column values."
    )

print(
    f"Complete tuning table created: "
    f"{TUNING_TABLE}"
)

print(
    f"Complete tuning rows: "
    f"{tuning_row_count:,}"
)

display(
    df_tuning_complete
    .groupBy(
        TARGET_COLUMN
    )
    .count()
    .orderBy(
        TARGET_COLUMN
    )
)


# ============================================================
# 5. Chronological fold definitions
# ============================================================

TUNING_FOLDS = cfg.TUNING_FOLDS

if not TUNING_FOLDS:
    raise ValueError(
        "No chronological tuning folds were configured."
    )

print(
    f"Chronological tuning folds: "
    f"{len(TUNING_FOLDS)}"
)


# ============================================================
# 6. Prediction evaluation function
# ============================================================

from utils.model_training import (
    evaluate_tuning_predictions,
)


# ============================================================
# 7. Fold-level training and evaluation
# ============================================================

def train_and_evaluate_fold(
    estimator,
    training_df: DataFrame,
    validation_df: DataFrame,
) -> dict[str, float]:
    """
    Train one estimator and evaluate one chronological fold.
    """
    if (
        training_df
        .limit(1)
        .count()
        == 0
    ):
        raise ValueError(
            "A chronological training fold "
            "contains zero rows."
        )

    if (
        validation_df
        .limit(1)
        .count()
        == 0
    ):
        raise ValueError(
            "A chronological validation fold "
            "contains zero rows."
        )

    training_start_time = (
        time.perf_counter()
    )

    fitted_model = estimator.fit(
        training_df
    )

    training_seconds = (
        time.perf_counter()
        - training_start_time
    )

    predictions = fitted_model.transform(
        validation_df
    )

    metrics = evaluate_tuning_predictions(
        predictions,
        TARGET_COLUMN,
    )

    metrics["TRAINING_SECONDS"] = float(
        training_seconds
    )

    return metrics


# ============================================================
# 8. Reusable Spark hyperparameter-tuning function
# ============================================================

def tune_model(
    model_name: str,
    estimator_builder: Callable,
    parameter_grid: list[dict],
    tuning_df: DataFrame = df_tuning_complete,
    tuning_folds: list[dict] = TUNING_FOLDS,
) -> DataFrame:
    """
    Tune one Spark model across chronological folds.

    Training sampling is applied only after the chronological
    training period has been defined.

    A reproducible uniform validation sample is created only
    after the chronological validation period has been defined.
    The same validation limit and random seed are used for
    Logistic Regression, Random Forest, and XGBoost.
    """
    if not parameter_grid:
        raise ValueError(
            "The parameter grid cannot be empty."
        )

    tuning_results = []

    for configuration_number, params in enumerate(
        parameter_grid,
        start=1,
    ):
        fold_metrics = []

        print(
            f"Evaluating {model_name} configuration "
            f"{configuration_number}/"
            f"{len(parameter_grid)}: "
            f"{params}"
        )

        for fold_number, fold in enumerate(
            tuning_folds,
            start=1,
        ):
            # ------------------------------------------------
            # Define chronological periods before sampling
            # ------------------------------------------------

            complete_training_df = tuning_df.filter(
                F.col("FL_DATE")
                <= F.to_date(
                    F.lit(
                        fold["train_end"]
                    )
                )
            )

            complete_validation_df = tuning_df.filter(
                F.col("FL_DATE").between(
                    F.to_date(
                        F.lit(
                            fold[
                                "validation_start"
                            ]
                        )
                    ),
                    F.to_date(
                        F.lit(
                            fold[
                                "validation_end"
                            ]
                        )
                    ),
                )
            )

            # ------------------------------------------------
            # Sample the chronological training period
            # ------------------------------------------------

            training_df = (
                complete_training_df
                .sampleBy(
                    col=TARGET_COLUMN,
                    fractions=(
                        cfg
                        .TREE_TUNING_SAMPLE_FRACTIONS
                    ),
                    seed=RANDOM_SEED,
                )
            )

            # ------------------------------------------------
            # Apply the shared validation limit
            # ------------------------------------------------

            validation_df = bounded_spark_sample(
                complete_validation_df,
                TUNING_VALIDATION_MAX_ROWS,
                stratified=False,
            )

            training_count = (
                training_df.count()
            )

            validation_count = (
                validation_df.count()
            )

            if training_count == 0:
                raise ValueError(
                    f"{model_name} training fold "
                    f"{fold_number} contains zero rows."
                )

            if validation_count == 0:
                raise ValueError(
                    f"{model_name} validation fold "
                    f"{fold_number} contains zero rows."
                )

            print(
                f"{model_name} fold {fold_number}: "
                f"{training_count:,} sampled training rows, "
                f"{validation_count:,} sampled validation rows"
            )

            # ------------------------------------------------
            # Train and evaluate this configuration
            # ------------------------------------------------

            estimator = estimator_builder(
                **params
            )

            metrics = train_and_evaluate_fold(
                estimator=estimator,
                training_df=training_df,
                validation_df=validation_df,
            )

            fold_metrics.append(
                metrics
            )

        fold_count = len(
            fold_metrics
        )

        if fold_count == 0:
            raise ValueError(
                f"No fold metrics were produced "
                f"for {model_name}."
            )

        tuning_results.append(
            {
                "MODEL": model_name,

                "PARAMETERS": str(
                    params
                ),

                "ACCURACY": round(
                    sum(
                        item["ACCURACY"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "PRECISION": round(
                    sum(
                        item["PRECISION"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "RECALL": round(
                    sum(
                        item["RECALL"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "F1_SCORE": round(
                    sum(
                        item["F1_SCORE"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "ROC_AUC": round(
                    sum(
                        item["ROC_AUC"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "PR_AUC": round(
                    sum(
                        item["PR_AUC"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "DELAY_PRECISION": round(
                    sum(
                        item["DELAY_PRECISION"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "DELAY_RECALL": round(
                    sum(
                        item["DELAY_RECALL"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "DELAY_F1": round(
                    sum(
                        item["DELAY_F1"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    4,
                ),

                "TRAINING_SECONDS": round(
                    sum(
                        item["TRAINING_SECONDS"]
                        for item in fold_metrics
                    )
                    / fold_count,
                    2,
                ),
            }
        )

    return (
        spark.createDataFrame(
            tuning_results
        )
        .select(
            "MODEL",
            "PARAMETERS",
            "ACCURACY",
            "PRECISION",
            "RECALL",
            "F1_SCORE",
            "ROC_AUC",
            "PR_AUC",
            "DELAY_PRECISION",
            "DELAY_RECALL",
            "DELAY_F1",
            "TRAINING_SECONDS",
        )
        .orderBy(
            F.desc("DELAY_RECALL"),
            F.desc("DELAY_F1"),
            F.desc("PR_AUC"),
            F.desc("ROC_AUC"),
            F.asc("TRAINING_SECONDS"),
        )
    )


print(
    "Chronological tuning framework created with "
    f"a shared validation limit of "
    f"{TUNING_VALIDATION_MAX_ROWS:,} rows per fold."
)

In [0]:
first_fold = TUNING_FOLDS[0]

complete_training_check = df_tuning_complete.filter(
    F.col("FL_DATE")
    <= F.to_date(F.lit(first_fold["train_end"]))
)

sampled_training_check = complete_training_check.sampleBy(
    col=TARGET_COLUMN,
    fractions=cfg.TREE_TUNING_SAMPLE_FRACTIONS,
    seed=RANDOM_SEED,
)

validation_check = df_tuning_complete.filter(
    F.col("FL_DATE").between(
        F.to_date(F.lit(first_fold["validation_start"])),
        F.to_date(F.lit(first_fold["validation_end"])),
    )
)

print("Complete training-fold distribution")
display(
    complete_training_check
    .groupBy(TARGET_COLUMN)
    .count()
    .orderBy(TARGET_COLUMN)
)

print("Sampled training-fold distribution")
display(
    sampled_training_check
    .groupBy(TARGET_COLUMN)
    .count()
    .orderBy(TARGET_COLUMN)
)

print("Untouched validation-fold distribution")
display(
    validation_check
    .groupBy(TARGET_COLUMN)
    .count()
    .orderBy(TARGET_COLUMN)
)

#### Logistic Regression Hyperparameter Tuning

Logistic Regression is optimized by evaluating multiple combinations of regularization strength and elastic-net mixing.

These hyperparameters control model complexity and reduce the risk of overfitting while maintaining good generalization performance.

Each candidate configuration is evaluated using the chronological validation folds defined previously. The average Accuracy, weighted Precision, weighted Recall, weighted F1-score, delayed-flight Precision, delayed-flight Recall, delayed-flight F1-score, ROC AUC, and Precision–Recall AUC are calculated for every configuration.

The configurations are compared using the complete metric set, with particular attention given to delayed-flight detection performance.


In [0]:
from pyspark.ml.classification import LogisticRegression


LOGISTIC_REGRESSION_PARAMETER_GRID = [
    {
        "regParam": 0.001,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.10,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 0.5,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 1.0,
    },
]

print(
    f"Candidate Logistic Regression configurations: "
    f"{len(LOGISTIC_REGRESSION_PARAMETER_GRID)}"
)

#### Logistic Regression Estimator Builder

A reusable estimator builder is created for Logistic Regression.

Rather than manually constructing a new Logistic Regression model for every hyperparameter configuration, this function generates a configured estimator using the supplied tuning parameters.

Only the hyperparameters under evaluation (`regParam` and `elasticNetParam`) vary between candidate configurations. All remaining model settings remain fixed to ensure that performance differences are attributable solely to the tuned parameters.

This reusable design allows the same chronological tuning workflow to evaluate every Logistic Regression configuration consistently while minimizing duplicated code.


In [0]:
def build_logistic_regression(
    regParam,
    elasticNetParam,
):

    return LogisticRegression(
        featuresCol="features",
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",

        regParam=regParam,
        elasticNetParam=elasticNetParam,

        maxIter=cfg.SELECTED_LR_MAX_ITER,
        standardization=True,
        family="binomial",
    )


print("Logistic Regression estimator builder created successfully.")

## Tune Logistic Regression

Five Logistic Regression hyperparameter configurations are evaluated using the shared chronological tuning framework. The configurations vary the regularization strength (`regParam`) and the balance between L1 and L2 regularization (`elasticNetParam`).

For each configuration:

1. The model is trained using four expanding-window chronological training folds.
2. Training-fold sampling is performed only after the chronological boundaries are established.
3. The model is evaluated on a separate, reproducible validation sample of approximately 50,000 rows from the immediately following month.
4. Validation sampling is uniform rather than class-stratified, preserving the month’s natural delayed-flight distribution.
5. Performance metrics are averaged across the four folds.

The configurations are compared using delayed-flight Precision, delayed-flight Recall, delayed-flight F1-score, ROC AUC, PR AUC, overall classification metrics, and training time. Delayed-flight Recall is treated as the primary ranking metric because failing to identify an actually delayed flight represents the main operational risk. Delayed-flight F1-score and PR AUC are also considered to account for the precision–recall trade-off.

In [0]:
lr_tuning_results = tune_model(
    model_name="Logistic Regression",
    estimator_builder=build_logistic_regression,
    parameter_grid=LOGISTIC_REGRESSION_PARAMETER_GRID,
)

display(lr_tuning_results)

## Logistic Regression Tuning Results

Five Logistic Regression configurations were evaluated using four chronological expanding-window validation folds. Each reported value represents the average performance across those folds using the shared validation-sampling procedure.

Under the delayed-flight Recall-first ranking policy, the best-performing configuration used:

- `regParam = 0.001`
- `elasticNetParam = 0.0`

This configuration achieved:

- Accuracy: **0.5585**
- Weighted Precision: **0.7292**
- Weighted Recall: **0.5585**
- Weighted F1-score: **0.5761**
- ROC AUC: **0.6862**
- PR AUC: **0.4081**
- Delayed-flight Precision: **0.3434**
- Delayed-flight Recall: **0.7602**
- Delayed-flight F1-score: **0.4681**
- Average training time: **9.83 seconds**

This configuration identified approximately **76.02% of delayed flights**, the highest delayed-flight Recall among the five Logistic Regression candidates. It also achieved the highest delayed-flight F1-score and PR AUC within the Logistic Regression search.

The results show a consistent precision–recall trade-off. Increasing the regularization strength generally increased Accuracy and delayed-flight Precision but reduced delayed-flight Recall. For example, the configuration using `regParam=0.01` and `elasticNetParam=1.0` achieved the highest Accuracy of **0.6227** and delayed-flight Precision of **0.3772**, but its delayed-flight Recall decreased to **0.6164** and its delayed-flight F1-score decreased to **0.4452**.

ROC AUC remained relatively stable across all configurations, ranging from **0.6826 to 0.6862**. This suggests that the configurations had similar ranking ability, while regularization primarily changed the balance between positive delay alerts and missed delayed flights at the default classification threshold.

Therefore, the weakly regularized L2 configuration with `regParam=0.001` and `elasticNetParam=0.0` is retained as the tuned Logistic Regression candidate. It will be compared with the strongest tuned Random Forest and XGBoost configurations before the final model is selected.


#### Random Forest Hyperparameter Tuning

Random Forest is optimized by evaluating multiple combinations of the number of trees and maximum tree depth.

The number of trees controls the size of the ensemble, while maximum depth controls the complexity of each individual decision tree. Increasing these values may improve predictive performance, but it also increases training time and the risk of overfitting.

Each candidate configuration is evaluated using the same four chronological expanding-window validation folds used for Logistic Regression.

The reported metrics represent the average performance across those folds. The configurations are compared using Accuracy, weighted Precision, weighted Recall, weighted F1-score, delayed-flight Precision, delayed-flight Recall, delayed-flight F1-score, ROC AUC, PR AUC, and average training time.


In [0]:
from pyspark.ml.classification import RandomForestClassifier


RANDOM_FOREST_PARAMETER_GRID = [
    {
        "numTrees": 5,
        "maxDepth": 4,
    },
    {
        "numTrees": 5,
        "maxDepth": 6,
    },
    {
        "numTrees": 10,
        "maxDepth": 4,
    },
    {
        "numTrees": 10,
        "maxDepth": 6,
    },
]


print(
    f"Candidate Random Forest configurations: "
    f"{len(RANDOM_FOREST_PARAMETER_GRID)}"
)

#### Random Forest Estimator Builder

A reusable estimator builder is created for Random Forest classification.

The function constructs a new Random Forest estimator for each combination of `numTrees` and `maxDepth` in the parameter grid.

Only the hyperparameters being evaluated vary between configurations. The remaining settings are held constant so that performance differences can be attributed to the selected number of trees and tree depth.

The estimator is designed to work with the same chronological tuning workflow used for Logistic Regression.


In [0]:
def build_random_forest(
    numTrees,
    maxDepth,
):
    """Build a configured Random Forest estimator."""

    return RandomForestClassifier(
        featuresCol="features",
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        numTrees=numTrees,
        maxDepth=maxDepth,
        seed=cfg.RANDOM_SEED,
    )


print("Random Forest estimator builder created successfully.")

## Tune Random Forest

Four Random Forest hyperparameter configurations are evaluated using the shared chronological tuning framework. The configurations vary the number of trees (`numTrees`) and the maximum depth of each tree (`maxDepth`).

For each configuration:

1. The model is trained across four expanding-window chronological training folds.
2. Training-fold sampling is performed only after the chronological boundaries are established.
3. The model is evaluated on a separate, reproducible validation sample of approximately 50,000 rows from the immediately following month.
4. Validation sampling is uniform, preserving the month’s natural delayed-flight distribution.
5. Performance metrics are averaged across the four folds.

The configurations are compared using overall classification metrics, delayed-flight Precision, delayed-flight Recall, delayed-flight F1-score, ROC AUC, PR AUC, and average training time. Delayed-flight Recall is used as the primary ranking metric, with delayed-flight F1-score and PR AUC providing additional measures of minority-class performance.


In [0]:
rf_tuning_results = tune_model(
    model_name="Random Forest",
    estimator_builder=build_random_forest,
    parameter_grid=RANDOM_FOREST_PARAMETER_GRID,
)

display(rf_tuning_results)

## Random Forest Tuning Results

Four Random Forest configurations were evaluated using four chronological expanding-window validation folds. Each reported value represents the average performance across those folds using the shared validation-sampling procedure.

Under the delayed-flight Recall-first ranking policy, the strongest Random Forest configuration used:

- `numTrees = 10`
- `maxDepth = 6`

This configuration achieved:

- Accuracy: **0.7050**
- Weighted Precision: **0.7255**
- Weighted Recall: **0.7050**
- Weighted F1-score: **0.6545**
- ROC AUC: **0.6719**
- PR AUC: **0.3975**
- Delayed-flight Precision: **0.5906**
- Delayed-flight Recall: **0.2375**
- Delayed-flight F1-score: **0.2144**
- Average training time: **75.79 seconds**

This configuration achieved the highest delayed-flight Recall, delayed-flight F1-score, ROC AUC, and PR AUC among the four Random Forest candidates. Increasing the tree depth from 4 to 6 generally improved the model’s ability to identify delayed flights by allowing the individual trees to learn more complex relationships.

The configuration with `numTrees=10` and `maxDepth=4` achieved the highest overall Accuracy of **0.7214** and weighted F1-score of **0.6605**. However, its delayed-flight Recall was only **0.1781**, meaning that it missed more than 82% of delayed flights. This demonstrates why overall Accuracy alone is not sufficient for selecting a model for this project.

Although the selected Random Forest configuration produced relatively strong delayed-flight Precision of **0.5906**, its delayed-flight Recall of **0.2375** remained substantially lower than the tuned Logistic Regression Recall of **0.7602**. Random Forest therefore generated fewer false-positive delay alerts, but it failed to identify most flights that were actually delayed.

The `numTrees=10`, `maxDepth=6` configuration is retained as the strongest tuned Random Forest candidate for comparison with tuned Logistic Regression and XGBoost during final candidate-model selection.


## XGBoost Hyperparameter Tuning

Standard XGBoost is tuned using a compact grid of four candidate configurations. The search varies tree depth, number of boosting rounds, and learning rate while keeping row and feature subsampling fixed.

The following hyperparameters are evaluated:

- `max_depth`: **4 or 6**
- `n_estimators`: **50 or 100**
- `learning_rate`: **0.10 or 0.05**
- `subsample`: fixed at **0.80**
- `colsample_bytree`: fixed at **0.80**

The four configurations combine shallower and deeper trees with two boosting strategies:

1. 50 trees, maximum depth 4, and learning rate 0.10.
2. 50 trees, maximum depth 6, and learning rate 0.10.
3. 100 trees, maximum depth 4, and learning rate 0.05.
4. 100 trees, maximum depth 6, and learning rate 0.05.

The 50-tree configurations use a higher learning rate to make larger updates over fewer boosting rounds. The 100-tree configurations use a lower learning rate to make smaller, more gradual updates. Tree depths of 4 and 6 are compared to determine whether additional model complexity improves delayed-flight detection.

Every configuration uses the same four expanding-window chronological folds. Within each fold:

- The chronological training period is defined before sampling.
- A reproducible, class-stratified training sample of approximately 50,000 rows is converted from Spark vectors into a SciPy sparse matrix.
- A separate, uniformly selected validation sample of approximately 50,000 rows is taken from the following month.
- The validation sample preserves the month’s natural class distribution.
- The same sampling limits and chronological boundaries used by the other tuned candidates are maintained for comparability.

The compact four-configuration grid keeps XGBoost tuning computationally practical within Databricks Free Edition while still testing meaningful differences in model complexity and boosting behavior.

In [0]:
XGBOOST_PARAMETER_GRID = [
    {
        "max_depth": 4,
        "n_estimators": 50,
        "learning_rate": 0.10,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
    },
    {
        "max_depth": 6,
        "n_estimators": 50,
        "learning_rate": 0.10,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
    },
    {
        "max_depth": 4,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
    },
    {
        "max_depth": 6,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
    },
]

print(
    "Candidate XGBoost configurations: "
    f"{len(XGBOOST_PARAMETER_GRID)}"
)

## XGBoost Estimator Builder

This section defines a reusable estimator builder that creates a fresh standard `XGBClassifier` for every candidate hyperparameter configuration.

The builder receives the configuration-specific values for:

- Maximum tree depth
- Number of boosting trees
- Learning rate
- Row subsampling
- Feature subsampling

The remaining settings are held constant across all configurations:

- `objective="binary:logistic"` configures XGBoost for binary flight-delay classification and produces probability estimates.
- `eval_metric="logloss"` evaluates the quality of the predicted class probabilities during training.
- `tree_method="hist"` uses histogram-based tree construction to reduce computational and memory requirements.
- `random_state=cfg.RANDOM_SEED` makes model training reproducible.
- `n_jobs=-1` allows XGBoost to use the available processor threads.

Each estimator is trained locally using the SciPy sparse matrices prepared from the bounded Spark training samples. This preserves the sparse hashed-feature representation and avoids constructing a dense pandas dataset in driver memory.

Creating a new estimator for every configuration and chronological fold also prevents fitted model state from carrying over between tuning experiments.

In [0]:
def build_xgboost(
    max_depth,
    n_estimators,
    learning_rate,
    subsample,
    colsample_bytree,
):
    """Build a configured standard XGBoost classifier."""

    return XGBClassifier(
        max_depth=max_depth,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=cfg.RANDOM_SEED,
        n_jobs=-1,
    )


print("Standard XGBoost estimator builder created successfully.")


## Verify XGBoost Estimator Compatibility

Before tuning begins, the first parameter-grid configuration is passed to the XGBoost estimator builder.

This lightweight compatibility check confirms that:

- The standard `XGBClassifier` package is available.
- The estimator builder accepts every required hyperparameter.
- The supplied parameter names are compatible with the installed XGBoost version.
- A correctly configured `XGBClassifier` object can be created in the Databricks environment.

This step only constructs the estimator. It does not fit a model or evaluate any data. Successful completion confirms that the tuning workflow can proceed with the defined parameter grid.

In [0]:
xgboost_compatibility_estimator = build_xgboost(
    **XGBOOST_PARAMETER_GRID[0]
)

print(
    "Standard XGBoost compatibility check passed: "
    f"{type(xgboost_compatibility_estimator).__name__}"
)


## Tune XGBoost

The four standard XGBoost configurations are evaluated using the same four expanding-window chronological folds and validation limit used for Logistic Regression and Random Forest.

For each fold, the tuning function:

1. Loads a fresh handle to the prepared tuning table to reduce the risk of an expired Databricks Serverless operation handle.
2. Defines the chronological training and validation periods before performing any sampling.
3. Draws a reproducible, class-stratified training sample of approximately 50,000 rows.
4. Draws a separate, uniformly selected validation sample of approximately 50,000 rows from the following month.
5. Converts the Spark feature vectors into SciPy sparse matrices for local XGBoost processing.
6. Reuses the prepared training and validation matrices across all four XGBoost configurations within that fold.
7. Creates and fits a fresh `XGBClassifier` for each configuration.
8. Evaluates predicted classes and probabilities using the same classification and ranking metrics reported for the Spark models.

The validation sample is not class-balanced. Uniform sampling preserves the validation month’s natural class distribution and provides a more realistic estimate of model performance.

For each configuration, the fold-level metrics are averaged across all four chronological folds. The returned results include:

- Accuracy
- Weighted Precision
- Weighted Recall
- Weighted F1-score
- ROC AUC
- PR AUC
- Delayed-flight Precision
- Delayed-flight Recall
- Delayed-flight F1-score
- Average training time

The returned schema matches the Logistic Regression and Random Forest tuning results, allowing the strongest configuration from each algorithm to be combined in a single candidate-model comparison.

XGBoost configurations are ranked primarily by delayed-flight Recall, followed by delayed-flight F1-score, PR AUC, and ROC AUC. Training time is retained as an additional measure of computational efficiency.

In [0]:
def tune_local_xgboost(
    parameter_grid,
    tuning_table=TUNING_TABLE,
    tuning_folds=TUNING_FOLDS,
):
    """
    Tune standard XGBoost using bounded chronological
    training and validation samples.

    The validation limit and sampling seed match the
    Logistic Regression and Random Forest workflow.
    """
    if not parameter_grid:
        raise ValueError(
            "The XGBoost parameter grid cannot be empty."
        )

    metrics_by_configuration = [
        [] for _ in parameter_grid
    ]

    for fold_number, fold in enumerate(
        tuning_folds,
        start=1,
    ):
        # Use a fresh Serverless table handle for this fold.
        fold_source_df = spark.table(
            tuning_table
        )

        # Define chronological periods before sampling.
        complete_training_df = fold_source_df.filter(
            F.col("FL_DATE")
            <= F.to_date(
                F.lit(fold["train_end"])
            )
        )

        complete_validation_df = fold_source_df.filter(
            F.col("FL_DATE").between(
                F.to_date(
                    F.lit(
                        fold["validation_start"]
                    )
                ),
                F.to_date(
                    F.lit(
                        fold["validation_end"]
                    )
                ),
            )
        )

        # Create the bounded training sample.
        local_training_df = bounded_spark_sample(
            complete_training_df,
            XGB_MAX_TRAIN_ROWS,
            stratified=True,
        )

        # Use the same bounded validation rule as LR and RF.
        local_validation_df = bounded_spark_sample(
            complete_validation_df,
            TUNING_VALIDATION_MAX_ROWS,
            stratified=False,
        )

        # Convert the bounded samples to local sparse matrices.
        X_train_fold, y_train_fold = (
            spark_vectors_to_csr(
                local_training_df
            )
        )

        X_validation_fold, y_validation_fold = (
            spark_vectors_to_csr(
                local_validation_df
            )
        )

        print(
            f"XGBoost fold {fold_number}: "
            f"{X_train_fold.shape[0]:,} sampled "
            f"training rows, "
            f"{X_validation_fold.shape[0]:,} sampled "
            f"validation rows"
        )

        for configuration_index, params in enumerate(
            parameter_grid
        ):
            print(
                f"Fold {fold_number}, "
                f"XGBoost configuration "
                f"{configuration_index + 1}/"
                f"{len(parameter_grid)}: "
                f"{params}"
            )

            estimator = build_xgboost(
                **params
            )

            training_start_time = (
                time.perf_counter()
            )

            estimator.fit(
                X_train_fold,
                y_train_fold,
            )

            training_seconds = (
                time.perf_counter()
                - training_start_time
            )

            metrics, _, _ = evaluate_local_xgboost(
                estimator,
                X_validation_fold,
                y_validation_fold,
            )

            metrics["TRAINING_SECONDS"] = float(
                training_seconds
            )

            metrics_by_configuration[
                configuration_index
            ].append(
                metrics
            )

            print(
                f"Completed fold {fold_number}, "
                f"configuration "
                f"{configuration_index + 1}: "
                f"delay recall="
                f"{metrics['DELAY_RECALL']:.4f}, "
                f"delay F1="
                f"{metrics['DELAY_F1']:.4f}, "
                f"PR AUC="
                f"{metrics['PR_AUC']:.4f}"
            )

            # Maintain Serverless session activity.
            spark.sql(
                "SELECT current_timestamp()"
            ).collect()

        # Release local matrices before preparing the next fold.
        del X_train_fold
        del y_train_fold
        del X_validation_fold
        del y_validation_fold

        del local_training_df
        del local_validation_df
        del complete_training_df
        del complete_validation_df
        del fold_source_df

    result_rows = []

    metric_columns = [
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
        "DELAY_PRECISION",
        "DELAY_RECALL",
        "DELAY_F1",
        "TRAINING_SECONDS",
    ]

    for params, fold_metrics in zip(
        parameter_grid,
        metrics_by_configuration,
    ):
        if len(fold_metrics) != len(
            tuning_folds
        ):
            raise ValueError(
                "XGBoost did not produce metrics "
                "for every chronological fold."
            )

        result = {
            "MODEL": "XGBoost",
            "PARAMETERS": str(params),
        }

        for metric_name in metric_columns:
            decimal_places = (
                2
                if metric_name == "TRAINING_SECONDS"
                else 4
            )

            result[metric_name] = round(
                sum(
                    fold_result[metric_name]
                    for fold_result in fold_metrics
                )
                / len(fold_metrics),
                decimal_places,
            )

        result_rows.append(
            result
        )

    return (
        spark.createDataFrame(
            result_rows
        )
        .select(
            "MODEL",
            "PARAMETERS",
            "ACCURACY",
            "PRECISION",
            "RECALL",
            "F1_SCORE",
            "ROC_AUC",
            "PR_AUC",
            "DELAY_PRECISION",
            "DELAY_RECALL",
            "DELAY_F1",
            "TRAINING_SECONDS",
        )
        .orderBy(
            F.desc("DELAY_RECALL"),
            F.desc("DELAY_F1"),
            F.desc("PR_AUC"),
            F.desc("ROC_AUC"),
            F.asc("TRAINING_SECONDS"),
        )
    )


xgb_tuning_results = tune_local_xgboost(
    XGBOOST_PARAMETER_GRID
)

display(
    xgb_tuning_results
)

## XGBoost Tuning Results

Four XGBoost hyperparameter configurations were evaluated using four chronological expanding-window validation folds. Each reported value represents the average performance across those folds using the same bounded training and validation-sampling policy applied during the tuned-model comparison.

Under the delayed-flight Recall-first ranking policy, the strongest XGBoost configuration used:

- `max_depth = 4`
- `n_estimators = 100`
- `learning_rate = 0.05`
- `subsample = 0.80`
- `colsample_bytree = 0.80`

This configuration achieved:

- Accuracy: **0.5828**
- Weighted Precision: **0.7296**
- Weighted Recall: **0.5828**
- Weighted F1-score: **0.6059**
- ROC AUC: **0.6928**
- PR AUC: **0.4146**
- Delayed-flight Precision: **0.3533**
- Delayed-flight Recall: **0.7469**
- Delayed-flight F1-score: **0.4785**
- Average training time: **0.82 seconds**

This configuration identified approximately **74.69% of delayed flights**, the highest delayed-flight Recall among the four XGBoost candidates. The differences among the configurations were small, indicating that XGBoost performance was relatively stable within the compact search space.

The shallower 50-tree configuration with `max_depth=4` and `learning_rate=0.10` produced nearly identical performance, including delayed-flight Recall of **0.7468** and delayed-flight F1-score of **0.4785**, while recording the lowest average training time of **0.47 seconds**.

The configuration using `max_depth=6`, `n_estimators=100`, and `learning_rate=0.05` achieved the highest overall Accuracy of **0.5860**, weighted F1-score of **0.6090**, ROC AUC of **0.6931**, PR AUC of **0.4165**, delayed-flight Precision of **0.3552**, and delayed-flight F1-score of **0.4796**. However, its delayed-flight Recall was slightly lower at **0.7445**.

Because the current tuning policy ranks configurations first by delayed-flight Recall, the shallower 100-tree configuration is retained as the tuned XGBoost candidate. Its Recall advantage over the deeper 100-tree configuration is small—approximately **0.24 percentage points**—so the two configurations should be regarded as operationally similar rather than meaningfully different.

Overall, increasing tree depth did not materially improve delayed-flight detection. The results suggest that shallower trees provide competitive performance with lower complexity, while additional boosting rounds combined with a lower learning rate produce the strongest delayed-flight Recall within the evaluated grid.

The tuned XGBoost candidate is retained for comparison with tuned Logistic Regression and Random Forest. Because all three tuning workflows use the same chronological folds, reproducible sampling policy, and approximate validation limit, their tuned performance can be compared under a consistent evaluation design.

#### Tuned Model Comparison

The strongest tuned configuration from Logistic Regression, Random Forest, and XGBoost is selected dynamically from each algorithm’s chronological tuning results.

The retained configurations are combined into a common comparison table using their average performance across the four expanding-window validation folds.

The comparison considers:

Overall Accuracy

Weighted Precision

Weighted Recall

Weighted F1-score

ROC AUC

PR AUC

Delayed-flight Precision

Delayed-flight Recall

Delayed-flight F1-score

Average training time

Because the primary operational objective is to identify flights likely to experience arrival delays of at least 15 minutes, the candidates are ranked first by delayed-flight Recall. Delayed-flight F1-score, PR AUC, ROC AUC, and training efficiency are used as subsequent criteria.

The comparison provides the basis for selecting the candidate final model that proceeds to retraining and evaluation on the untouched holdout period. The tuning-fold averages support candidate selection but are not treated as the final estimate of future predictive performance.


In [0]:
from pyspark.sql import functions as F


def best_tuned_configuration(results):
    """Return the top row under the shared operational ranking."""
    return results.orderBy(
        F.desc("DELAY_RECALL"),
        F.desc("DELAY_F1"),
        F.desc("PR_AUC"),
        F.desc("ROC_AUC"),
        F.asc("TRAINING_SECONDS"),
    ).limit(1)


best_logistic_regression = best_tuned_configuration(
    lr_tuning_results
)
best_random_forest = best_tuned_configuration(
    rf_tuning_results
)
best_xgboost = best_tuned_configuration(
    xgb_tuning_results
)

tuned_model_comparison = (
    best_logistic_regression
    .unionByName(best_random_forest)
    .unionByName(best_xgboost)
)

ranked_model_comparison = tuned_model_comparison.orderBy(
    F.desc("DELAY_RECALL"),
    F.desc("DELAY_F1"),
    F.desc("PR_AUC"),
    F.desc("ROC_AUC"),
    F.asc("TRAINING_SECONDS"),
)

display(ranked_model_comparison)


## Tuned Model Comparison Results

The strongest configuration from Logistic Regression, Random Forest, and XGBoost was compared using average performance across the four chronological validation folds.

The comparison produced the following findings:

- **Logistic Regression** achieved the highest delayed-flight Recall of **0.7602**. It identified approximately 76.02% of delayed flights, but achieved delayed-flight Precision of **0.3434** and delayed-flight F1-score of **0.4681**. Its ROC AUC was **0.6862**, PR AUC was **0.4081**, and average training time was **9.83 seconds**.

- **XGBoost** achieved delayed-flight Recall of **0.7469**, only **0.0133** lower than Logistic Regression. However, XGBoost produced the highest delayed-flight F1-score of **0.4785**, the highest ROC AUC of **0.6928**, the highest PR AUC of **0.4146**, and slightly higher delayed-flight Precision of **0.3533**. It also recorded the lowest average training time at **0.82 seconds**.

- **Random Forest** achieved the highest overall Accuracy of **0.7050**, weighted F1-score of **0.6545**, and delayed-flight Precision of **0.5906**. However, its delayed-flight Recall was only **0.2375**, meaning that it failed to identify more than three-quarters of delayed flights. Its delayed-flight F1-score of **0.2144** was also substantially lower than those of Logistic Regression and XGBoost.

These results demonstrate the trade-off between overall classification performance and delayed-flight detection. Random Forest generated more reliable positive delay alerts, but its low Recall makes it unsuitable when identifying as many delayed flights as possible is the primary objective.

Logistic Regression achieved the highest delayed-flight Recall, but its advantage over XGBoost was only **1.33 percentage points**. XGBoost performed better on delayed-flight F1-score, delayed-flight Precision, ROC AUC, PR AUC, overall Accuracy, weighted F1-score, and training efficiency.

Under the notebook’s current strict Recall-first ranking rule, **Logistic Regression is ranked first** because its delayed-flight Recall of **0.7602** is higher than XGBoost’s **0.7469**. This selection is consistent with the implemented ranking logic.

However, the comparison also indicates that **XGBoost provides the strongest balanced performance**. If models within approximately one to two percentage points of the highest Recall are considered operationally equivalent, XGBoost would be the preferred candidate because it provides stronger F1-score and probability-ranking performance while retaining nearly the same delayed-flight Recall.

A limitation of the comparison is that all models use the same chronological validation design and approximately 50,000 validation observations per fold, but standard XGBoost is trained on a driver-safe sample of approximately 50,000 rows per fold. Logistic Regression and Random Forest use substantially larger Spark training samples. The evaluation is therefore aligned on validation conditions, although the algorithms do not receive identical quantities of training data.

For consistency with the currently implemented selection policy, Logistic Regression is retained as the candidate final model. XGBoost remains a highly competitive alternative whose more balanced performance should be acknowledged when interpreting the final selection.


## Evaluate Candidate-Model Performance

### Purpose

This stage compares the strongest tuned configuration from Logistic Regression, Random Forest, and XGBoost. The values represent average performance across the four expanding-window chronological validation folds; they are used for candidate-model selection rather than final holdout evaluation.

The primary operational objective is to identify flights likely to arrive at least 15 minutes late before departure. Missing an actually delayed flight may prevent operational teams from taking proactive action. Delayed-flight Recall therefore receives greater emphasis than overall Accuracy.

The comparison considers:

- Overall Accuracy
- Weighted Precision
- Weighted Recall
- Weighted F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)
- Delayed-flight Precision
- Delayed-flight Recall
- Delayed-flight F1-score
- Average training time

The metrics serve different purposes:

- **Delayed-flight Recall** measures the proportion of actual delayed flights successfully identified.
- **Delayed-flight Precision** measures how often a delay alert is correct.
- **Delayed-flight F1-score** summarizes the balance between delayed-flight Precision and Recall.
- **PR AUC** evaluates probability-ranking performance for the minority delayed-flight class.
- **ROC AUC** measures the model’s general ability to distinguish delayed from on-time flights across classification thresholds.
- **Accuracy and weighted metrics** describe overall classification performance but may be influenced by the larger number of on-time flights.
- **Training time** provides additional information about computational efficiency.

The currently implemented ranking policy is lexicographic and Recall-first. Models are ordered by:

1. Highest delayed-flight Recall.
2. Highest delayed-flight F1-score.
3. Highest PR AUC.
4. Highest ROC AUC.
5. Lowest average training time.

Under this strict policy, any improvement in delayed-flight Recall takes priority over all subsequent metrics, even when the difference is small. The remaining metrics are used only when the preceding metric is tied.

The three candidates use the same chronological validation periods and approximately 50,000 uniformly sampled validation observations per fold. However, standard XGBoost uses a smaller driver-safe training sample than the Spark Logistic Regression and Random Forest models. This computational constraint should be acknowledged when interpreting the comparison.

After the candidate model is selected, it must still be retrained using its chosen hyperparameters and evaluated on the project’s untouched final holdout period. The final holdout results—not the tuning-fold averages—provide the project’s definitive estimate of future predictive performance.


In [0]:
display(tuned_model_comparison)

## Candidate-Model Evaluation Summary

The tuned-model comparison demonstrates that each algorithm provides a different balance between delayed-flight detection, false-positive control, overall classification performance, and computational efficiency.

**Logistic Regression** achieved the highest delayed-flight Recall of **0.7602**, making it the strongest model for maximizing the proportion of actual delayed flights identified. However, XGBoost followed closely with Recall of **0.7469**, a difference of only **0.0133**.

**Random Forest** achieved the highest overall Accuracy of **0.7050**, weighted F1-score of **0.6545**, and delayed-flight Precision of **0.5906**. However, its delayed-flight Recall of **0.2375** indicates that it missed most delayed flights.

**XGBoost** achieved the highest delayed-flight F1-score of **0.4785**, ROC AUC of **0.6928**, PR AUC of **0.4146**, and weighted Precision of **0.7296**. It also recorded the lowest average training time of **0.82 seconds**. These results indicate that XGBoost provides the strongest overall balance between delay detection, probability-ranking performance, and computational efficiency.

Under the notebook’s strict Recall-first selection policy, Logistic Regression ranks first. Nevertheless, XGBoost should be recognized as the strongest balanced candidate because it performs better across most secondary metrics while sacrificing only 1.33 percentage points of delayed-flight Recall.

## Model Strengths and Limitations

### Logistic Regression

**Strengths**

- Highest delayed-flight Recall at **0.7602**.
- Identifies approximately 76% of actual delayed flights.
- Simple and interpretable linear modeling approach.
- Provides coefficients that can support feature-level interpretation.
- Faster to train than Random Forest.

**Limitations**

- Lowest overall Accuracy at **0.5585**.
- Delayed-flight Precision of only **0.3434**, resulting in a relatively large number of false-positive delay alerts.
- Lower delayed-flight F1-score, ROC AUC, and PR AUC than XGBoost.
- Its Recall advantage over XGBoost is relatively small.
- Assumes a primarily linear relationship between the transformed predictors and the log-odds of delay.

### Random Forest

**Strengths**

- Highest overall Accuracy at **0.7050**.
- Highest weighted F1-score at **0.6545**.
- Highest delayed-flight Precision at **0.5906**.
- Captures nonlinear relationships and interactions between predictors.
- Produces fewer false-positive delay alerts than Logistic Regression and XGBoost.

**Limitations**

- Lowest delayed-flight Recall at **0.2375**.
- Misses approximately 76% of actual delayed flights.
- Lowest delayed-flight F1-score at **0.2144**.
- Lowest ROC AUC and PR AUC among the tuned candidates.
- Longest average training time at **75.79 seconds**.
- Its strong Accuracy is influenced by better performance on the majority on-time class.

### XGBoost

**Strengths**

- Highest delayed-flight F1-score at **0.4785**.
- Highest ROC AUC at **0.6928**.
- Highest PR AUC at **0.4146**.
- Highest weighted Precision at **0.7296**.
- Delayed-flight Recall of **0.7469**, close to Logistic Regression’s **0.7602**.
- Captures nonlinear relationships and predictor interactions.
- Lowest recorded average training time at **0.82 seconds**.
- Provides the strongest balance across delayed-flight detection and probability-ranking metrics.

**Limitations**

- Delayed-flight Recall is **0.0133** lower than Logistic Regression.
- Delayed-flight Precision remains relatively low at **0.3533**, resulting in many false-positive delay alerts.
- Less directly interpretable than Logistic Regression.
- Standard XGBoost is trained locally on a driver-safe sample of approximately 50,000 rows per fold, whereas the Spark models use larger training samples.
- Its recorded training time excludes Spark sampling and sparse-matrix preparation and therefore should not be interpreted as the complete end-to-end runtime.

## Candidate Selection Conclusion

If the project requires the maximum possible delayed-flight Recall regardless of small differences in other metrics, **Logistic Regression is the appropriate selection** and is correctly ranked first by the current code.

If models within approximately one to two percentage points of the highest Recall are considered operationally comparable, **XGBoost is the stronger balanced choice** because it achieves better delayed-flight F1-score, ROC AUC, PR AUC, overall Accuracy, weighted F1-score, and computational fit time while retaining nearly the same Recall.

Random Forest is not recommended as the final candidate because its higher Accuracy and delayed-flight Precision come at the cost of missing most delayed flights.

Following the notebook’s currently implemented strict Recall-first policy, Logistic Regression is retained as the candidate final model. The final decision remains subject to evaluation on the untouched holdout period.


## Select the Candidate Final Model

### Candidate-Model Selection Criteria

The candidate final model is selected dynamically from the ranked tuned-model comparison rather than being hard-coded to a particular algorithm.

The selection policy is aligned with the project’s operational objective of identifying flights likely to arrive at least 15 minutes late before departure. The models are ranked using the following criteria:

1. Highest delayed-flight Recall.
2. Highest delayed-flight F1-score.
3. Highest PR AUC.
4. Highest ROC AUC.
5. Lowest average training time.

Delayed-flight Recall is the primary criterion because missing an actually delayed flight reduces the opportunity for proactive operational intervention. Delayed-flight F1-score, PR AUC, and ROC AUC provide additional evidence about classification balance and probability-ranking performance.

Under this strict lexicographic policy, the tuned Logistic Regression model is selected because it achieved the highest delayed-flight Recall of **0.7602**. XGBoost followed closely with Recall of **0.7469** and achieved stronger delayed-flight F1-score, ROC AUC, and PR AUC, but these secondary metrics do not override Logistic Regression’s Recall advantage under the implemented ranking rule.

The selected Logistic Regression configuration uses:

- `regParam = 0.001`
- `elasticNetParam = 0.0`

This selection identifies the candidate model that proceeds to final retraining and evaluation on the untouched holdout period. The chronological tuning metrics support candidate selection but are not treated as the final estimate of future predictive performance.

In [0]:
import ast

from pyspark.sql import functions as F


selected_model_row = ranked_model_comparison.first()

if selected_model_row is None:
    raise ValueError(
        "The ranked tuned-model comparison contains no results."
    )

SELECTED_MODEL_NAME = selected_model_row["MODEL"]
SELECTED_MODEL_PARAMETERS = ast.literal_eval(
    selected_model_row["PARAMETERS"]
)

selected_model_summary = (
    ranked_model_comparison
    .filter(
        F.col("MODEL") == SELECTED_MODEL_NAME
    )
    .limit(1)
)

display(selected_model_summary)

print(f"Selected model: {SELECTED_MODEL_NAME}")
print(
    "Selected parameters: "
    f"{SELECTED_MODEL_PARAMETERS}"
)
print(
    "Selection policy: highest delayed-flight Recall, "
    "followed by delayed-flight F1-score, PR AUC, "
    "ROC AUC, and training efficiency."
)

## Candidate Final Model Selection Result

The dynamic selection process identified **Logistic Regression** as the candidate final model.

The selected hyperparameters are:

- `regParam = 0.001`
- `elasticNetParam = 0.0`

Across the four chronological validation folds, this configuration achieved:

- Accuracy: **0.5585**
- Weighted Precision: **0.7292**
- Weighted Recall: **0.5585**
- Weighted F1-score: **0.5761**
- ROC AUC: **0.6862**
- PR AUC: **0.4081**
- Delayed-flight Precision: **0.3434**
- Delayed-flight Recall: **0.7602**
- Delayed-flight F1-score: **0.4681**

Logistic Regression was selected because it achieved the highest delayed-flight Recall among the three tuned candidate algorithms. It identified approximately **76.02% of delayed flights**, compared with **74.69%** for XGBoost and **23.75%** for Random Forest.

XGBoost achieved stronger delayed-flight F1-score, ROC AUC, PR AUC, and overall classification performance than Logistic Regression. However, the notebook’s strict Recall-first selection policy gives priority to Logistic Regression’s **1.33-percentage-point Recall advantage**.

The selected configuration now proceeds to final model retraining and evaluation on the untouched holdout period. The final holdout results will determine whether the candidate’s delayed-flight detection performance generalizes to future observations.

Gradient-boosted trees are implemented with Spark ML `GBTClassifier` to stay inside the scalable Spark pipeline used for production scoring. This satisfies the tree-based boosting comparison requirement while keeping training, tuning, and deployment in a single Databricks workflow.

## Persist Modeling Checkpoints

This section saves the datasets and metadata required by the downstream model-evaluation and explainability notebooks.

Before saving, the historical modeling frames are cleaned so that only the stable model-input schema is retained. A validated feature hasher is then recreated and applied consistently to the training, validation, and test datasets.

The following Delta-table checkpoints are persisted:

- Hashed training dataset
- Hashed validation dataset
- Hashed test dataset
- Historical training dataset
- Historical validation dataset
- Historical test dataset
- Tuned-model comparison results

The checkpoint tables are written in overwrite mode with schema replacement enabled. This ensures that rerunning the notebook updates the saved artifacts using the current feature-processing logic and model-selection results.

A feature-manifest JSON file is also created. It records:

- The ordered model-input columns
- The feature-hashing configuration
- The dynamically selected model name
- The selected hyperparameters

A separate candidate-selection JSON file records:

- The selected candidate algorithm
- The selected hyperparameters
- The chronological tuning process as the source of the selection
- The stage at which the model was selected

The dynamically selected candidate is currently **Logistic Regression** with:

- `regParam = 0.001`
- `elasticNetParam = 0.0`

Cell 120 saves the selection metadata but does not persist a fitted Logistic Regression model. The downstream evaluation workflow uses these saved parameters and datasets to retrain the selected algorithm and evaluate it on the untouched holdout period.

Persisting the feature manifest and candidate-selection metadata ensures that downstream notebooks use the same input schema, feature transformation, selected algorithm, and tuned hyperparameters established during model training.

In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

import json

from utils.model_training import (
    build_feature_manifest,
    create_feature_hasher,
    hash_modeling_frame,
    prepare_hist_modeling_frame,
    validate_feature_hasher,
)

# Persist only the stable modeling schema without intermediate prior-count columns.
df_train_hist = prepare_hist_modeling_frame(df_train_hist)
df_validation_hist = prepare_hist_modeling_frame(df_validation_hist)
df_test_hist = prepare_hist_modeling_frame(df_test_hist)

# Rebuild hashed checkpoints from the cleaned hist tables before saving.
feature_hasher = create_feature_hasher()
validate_feature_hasher(feature_hasher)

df_train_hashed = hash_modeling_frame(df_train_hist, feature_hasher)
df_validation_hashed = hash_modeling_frame(df_validation_hist, feature_hasher)
df_test_hashed = hash_modeling_frame(df_test_hist, feature_hasher)

checkpoint_tables = [
    (cfg.MODELING_TRAIN_HASHED_TABLE, df_train_hashed),
    (cfg.MODELING_VALIDATION_HASHED_TABLE, df_validation_hashed),
    (cfg.MODELING_TEST_HASHED_TABLE, df_test_hashed),
    (cfg.MODELING_TRAIN_HIST_TABLE, df_train_hist),
    (cfg.MODELING_VALIDATION_HIST_TABLE, df_validation_hist),
    (cfg.MODELING_TEST_HIST_TABLE, df_test_hist),
]

for table_name, dataframe in checkpoint_tables:
    row_count = dataframe.count()
    print(f"Saving {table_name}: {row_count:,} rows")
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

(
    tuned_model_comparison.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.TUNED_MODEL_COMPARISON_TABLE)
)

feature_manifest = build_feature_manifest(
    selected_model_name=SELECTED_MODEL_NAME,
    selected_model_parameters=SELECTED_MODEL_PARAMETERS,
)

candidate_selection = {
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
    "selection_source": "Average performance across chronological tuning folds",
    "selected_after_hyperparameter_tuning": True,
}

dbutils.fs.put(
    cfg.MODEL_FEATURE_MANIFEST_PATH,
    json.dumps(feature_manifest, indent=4),
    overwrite=True,
)

dbutils.fs.put(
    cfg.CANDIDATE_SELECTION_PATH,
    json.dumps(candidate_selection, indent=4),
    overwrite=True,
)

print("Modeling checkpoints saved successfully.")
print(f"Feature manifest: {cfg.MODEL_FEATURE_MANIFEST_PATH}")
print(f"Candidate selection: {cfg.CANDIDATE_SELECTION_PATH}")
print(f"Manifest model inputs: {feature_manifest['model_input_columns']}")


In [0]:
import json


candidate_selection_saved = json.loads(
    dbutils.fs.head(
        cfg.CANDIDATE_SELECTION_PATH,
        10_000,
    )
)

print(
    json.dumps(
        candidate_selection_saved,
        indent=4,
    )
)

assert (
    candidate_selection_saved["selected_model_name"]
    == SELECTED_MODEL_NAME
)

assert (
    candidate_selection_saved[
        "selected_model_parameters"
    ]
    == SELECTED_MODEL_PARAMETERS
)

print("Candidate-selection metadata verified.")